# U-Net Forest / Non-Forest Mask Modelling

## What this notebook does
This Google Colab notebook builds a pixel-level forest/non-forest segmentation model from canopy-height labels and co-registered predictor rasters. It can operate on one area or several areas, select or reuse predictor channels, calculate normalization statistics, mine balanced image patches, train a configurable U-Net-family model, calibrate a classification threshold, evaluate validation and test data, and produce seamless full-scene probability rasters.

## Required inputs
- A top-level `DATA_DIR` containing the canopy-height model and, where used, LiDAR footprint rasters.
- A predictor subdirectory for each area containing co-registered single-band GeoTIFF predictor layers.
- For multi-area runs, consistently named rasters and predictor folders matching the templates in `CONFIG`.
- Optional previously generated `selected_features_<AREA>.json` files in `OUT_DIR`; otherwise the notebook can calculate a predictor subset.

## Outputs
- A best-performing Keras model named from `SAVE_EXPERIMENT_TAG` and saved in `OUT_DIR`.
- Selected-feature JSON files and, for multi-area runs, an optional common-feature JSON file.
- Normalization statistics saved as JSON when global multi-area statistics are calculated.
- Temporary training, validation, and test patch arrays stored under `/content/patch_cache` during the Colab session.
- Validation and independent-test metrics printed in the notebook, including ROC-AUC, PR-AUC, Brier score, accuracy, precision, recall, F1, IoU, MCC, and confusion-matrix counts.
- One forest-probability GeoTIFF for single-area mode or one probability GeoTIFF per area in multi-area mode.

## Settings a new user should review
- `CONFIG["DATA_DIR"]`: set the Google Drive folder containing the raster data.
- `AREA_TOKEN`, `CHM_FILENAME`, and `PREDICTOR_SUBDIR`: set these for the reference or single-area dataset.
- `USE_MULTI_AREA` and `AREA_CODES`: choose single-area or multi-area processing and list the required area codes.
- `CHM_TEMPLATE`, `PREDICTOR_SUBDIR_TEMPLATE`, and `LIDAR_FOOTPRINT_TEMPLATE`: change these if your naming convention differs.
- `OUT_DIR`, `SAVE_EXPERIMENT_TAG`, and `SAVE_PRED_TIF`: choose the output folder and filenames.
- Review `RGB_ONLY`, patch size and stride, split proportions, model variant, pretrained encoder settings, batch size, learning rate, epoch count, and evaluation threshold protocol before a new experiment.

## Important data assumptions
- All input rasters for an area must be co-registered and have matching dimensions.
- The current forest label is generated where the CHM is greater than zero; change that expression if a different tree-height threshold defines forest.
- Multi-area mode requires the expected LiDAR footprint raster for each included area and skips areas with missing mandatory inputs.
- The notebook installs pinned package versions and is designed primarily for Google Colab.

Run the notebook from top to bottom in Google Colab. The progress messages explain what is happening and where outputs are written.

## 1. Install the pinned runtime packages
Run this cell at the start of a fresh Colab session. Colab may request a runtime restart after package installation.

In [ ]:
!pip install --no-cache-dir --force-reinstall numpy==1.26.4 rasterio==1.3.9 tifffile==2024.2.12 scikit-image==0.24.0 tensorflow==2.16.1 tqdm==4.66.4
!pip install -U "ml-dtypes>=0.5.0"

## Prepare the runtime before importing numerical libraries

In [ ]:
# ================================================================
# Forest / Non-forest UNet - Colab Single-Cell (I/O-hardened + Sequence)
# ================================================================

# ---- Thread pinning BEFORE importing numpy/tf (reduces backend hiccups) ----
import os as _os
_os.environ["OMP_NUM_THREADS"] = "1"
_os.environ["OPENBLAS_NUM_THREADS"] = "1"
_os.environ["MKL_NUM_THREADS"] = "1"
_os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
_os.environ["NUMEXPR_NUM_THREADS"] = "1"
_os.environ["TF_NUM_INTEROP_THREADS"] = "1"
_os.environ["TF_NUM_INTRAOP_THREADS"] = "1"

## Set thread limits and import packages

In [ ]:
# -------------------------
# 0) Imports / Setup
# -------------------------
import os, json, random, shutil, gc, time
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import Window
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import Sequence
from keras.applications import ResNet50, EfficientNetB0, EfficientNetB2

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, jaccard_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    matthews_corrcoef, confusion_matrix,
)

## Mount Google Drive

In [ ]:
# -------------------------
# 1) Google Drive (Colab)
# -------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except Exception as e:
    print("Not in Colab or Drive mount skipped:", e)

## Define runtime helper functions

In [ ]:
# -------------------------
# 2) Runtime toggles (CPU-safe)
# -------------------------
def _maybe_enable_xla(flag=False):
    try:
        tf.config.optimizer.set_jit(bool(flag))
        print("XLA JIT:", bool(flag))
    except Exception as e:
        print("Could not set XLA JIT (ok):", e)

def _maybe_enable_mixed_precision(flag=False):
    try:
        from tensorflow.keras import mixed_precision
        mixed_precision.set_global_policy('mixed_float16' if flag else 'float32')
        print("Mixed precision:", "ON" if flag else "OFF")
    except Exception as e:
        print("Mixed precision not set (ok):", e)

## Configure the experiment
**This is the main cell a new user must edit.** At minimum, check `DATA_DIR`, the area identifiers and file templates, the output directory, and whether `USE_MULTI_AREA` is correct.

In [ ]:
# -------------------------
# 3) Configuration (CPU-friendly defaults)
# -------------------------
CONFIG = {
    # --- Paths / area control ---
    "DATA_DIR": "/content/drive/My Drive/canopy_height_models/",  # EDIT THIS PATH
    "AREA_TOKEN": "CF15",
    "CHM_FILENAME": "CF15_CHM.tif",          # original working CHM for single-area mode
    "PREDICTOR_SUBDIR": "CF15_layers",

    # --- Multi-area support ---
    "USE_MULTI_AREA": False,                # False = original single-area behaviour
    "MULTI_USE_COMMON_SELECTED_FEATURES": True,  # intersect per-area selected_features_*.json
    "AREA_CODES": ["N22","AE25","AW21","BF21","BO16","CF15","CR30","DB31"],
    "CHM_TEMPLATE": "{area}_CHM.tif",       # e.g. "AE25_CHM.tif"
    "PREDICTOR_SUBDIR_TEMPLATE": "{area}_layers",
    "LIDAR_FOOTPRINT_TEMPLATE": "{area}_lidar_footprint_full.tif",
    "LiDAR_MIN_COVER_FRAC": 0.99,           # keep patches with >= 99% LiDAR coverage
    "PRINT_LIDAR_STATS": True,

    # --- Input channel toggle ---
    "RGB_ONLY": False,                 # True => use ONLY RGB predictor rasters (red/green/blue)
    "RGB_BASENAMES": ["red.tif", "green.tif", "blue.tif"],  # expected filenames (case-insensitive)
    "RECOMPUTE_NORM_STATS_EACH_RUN": True,

    # --- Caching / I/O ---
    "USE_LOCAL_CACHE": True,
    "LOCAL_CACHE_DIR": "/content/predictor_cache",
    "READ_RETRIES": 2,
    "CLEAR_PATCH_CACHE_EACH_RUN": True,

    # --- Correlation pruning (DISABLED: reuse JSON only) ---
    "NUM_SAMPLES_FOR_CORR": 15000,
    "CORR_ABS_THRESHOLD": 0.90,

    # --- Normalization ---
    "NUM_SAMPLES_FOR_NORM": 20000,
    "SAVE_NORM_STATS_JSON": "norm_stats.json",

    # --- Iterative prune knobs (kept simple) ---
    "ITER_PRUNE_ENABLE": True,
    "ITER_PRUNE_WARMUP_EPOCHS": 2,
    "ITER_PRUNE_KEEP_K": 96,  # default 96
    "ITER_PRUNE_MIN_CHANNELS": 48,  # default 48
    "PRUNE_KEEP_IDX": None,
    "EPOCHS_AFTER_PRUNE": 100,

    # --- UNet shape/regularization ---
    "BASE_FILTERS": 16,
    "BOTTLENECK_CHANNELS": 64,
    "L2_REG": 1e-5,
    "BOTTLENECK_L1": 1e-5,
    "USE_SE": True,

    # --- Training patches ---
    "PATCH_SIZE": 64,   # default 96
    "PATCH_STRIDE": 32,  # default 96
    "POS_FRAC_THRESH": 0.40,  # default 0.30
    "BALANCE_POS_NEG": False,
    "TRAIN_RATIO": 0.80,
    "VAL_RATIO": 0.10,   # for early stopping + threshold calibration
    "TEST_RATIO": 0.10,  # final, untouched evaluation
    "MAX_TEST_PATCHES": 4000,
    "MAX_TRAIN_PATCHES": 20000,
    "MAX_VAL_PATCHES": 4000,

    # --- Model Optimization Toggles ---
    "USE_GROUPNORM": True,
    "USE_ASPP": True,
    "UPSAMPLE_MODE": "transpose",
    "LOSS_MODE": "focal_tversky",
    "USE_DUAL_CONTEXT": False,      # ASPP + PPM together at bottleneck
    "DEEP_SUPERVISION": True,       # multi-output at 4 decoder scales (upsampled to full size)
    "DS_LOSS_WEIGHTS": [1.0, 0.25, 0.1, 0.05],  # main,d2,d3,d4
    "USE_CBAM": False,              # CBAM attention replaces SE inside conv_block
    "CBAM_RATIO": 8,               # channel squeeze ratio for CBAM
    "MODEL_VARIANT": "unet3p",
    "USE_TRANSFORMER_BOTTLENECK": False,
    "TRANSFORMER_NUM_HEADS": 4,
    "TRANSFORMER_KEY_DIM": 32,
    "TRANSFORMER_MLP_MULT": 2,
    "USE_PRETRAINED_ENCODER": True,
    "ENCODER_BACKBONE": "resnet50",
    "FREEZE_ENCODER": False,

    # --- Optim/training ---
    "BATCH_SIZE": 2,
    "LEARNING_RATE": 3e-4,
    "AUGMENT": True,
    "INFER_THRESHOLD": 0.5,

    # --- Runtime toggles ---
    "ENABLE_MIXED_PRECISION": False,
    "ENABLE_XLA": False,
    "DEBUG_RUN_EAGERLY": False,
    "RETRY_SAFE_TRAINING_ON_ERROR": True,
    "SAFE_RETRY_BATCH_DIVISOR": 2,
    "DISABLE_MP_ON_RETRY": True,

    # --- Seamless prediction ---
    "PRED_OVERLAP_FRACTION": 0.5,

    # --- Evaluation threshold protocol ---
    "EVAL_THRESHOLD_MODE": "calibrated",  # "fixed" or "calibrated"
    "FIXED_THRESHOLD": 0.5,              # used when EVAL_THRESHOLD_MODE == "fixed"
    "VAL_CALIBRATION_STEPS": 200,        # more stable calibration, if you use "calibrated"

    # --- Outputs ---
    "OUT_DIR": "/content/drive/My Drive/unet_trial_outputs/",
    "SAVE_SELECTED_FEATURES_JSON": "selected_features.json",
    "SAVE_EXPERIMENT_TAG": "unet_final_CF15",
    "SAVE_PRED_TIF": "test_predictions.tif",
}

Path(CONFIG["OUT_DIR"]).mkdir(parents=True, exist_ok=True)
USE_MULTI = bool(CONFIG.get("USE_MULTI_AREA", False))
AREA_TOKEN = CONFIG["AREA_TOKEN"]

_maybe_enable_xla(CONFIG["ENABLE_XLA"])
_maybe_enable_mixed_precision(CONFIG["ENABLE_MIXED_PRECISION"])
try:
    tf.config.run_functions_eagerly(bool(CONFIG["DEBUG_RUN_EAGERLY"]))
    print("run_functions_eagerly:", CONFIG["DEBUG_RUN_EAGERLY"])
except Exception as e:
    print("Could not set eager flag (ok):", e)


def maybe_clear_patch_cache():
    if not CONFIG.get("CLEAR_PATCH_CACHE_EACH_RUN", False):
        return
    cache_dir = "/content/patch_cache"
    if os.path.exists(cache_dir):
        try:
            shutil.rmtree(cache_dir)
            print(f"Cleared patch cache: {cache_dir}")
        except Exception as e:
            print(f"Could not clear patch cache ({cache_dir}):", e)
    os.makedirs(cache_dir, exist_ok=True)

maybe_clear_patch_cache()

## Discover the required input files

In [ ]:
# -------------------------
# 4) Discover files
# -------------------------
DATA_DIR = Path(CONFIG["DATA_DIR"])
PRED_DIR = DATA_DIR / CONFIG["PREDICTOR_SUBDIR"]

def find_top_level_tifs(root: Path):
    return sorted([str(p) for p in root.glob("*.tif")])

def find_predictors_in_subdir(pred_root: Path):
    if not pred_root.exists():
        raise FileNotFoundError(f"Predictor subfolder not found: {pred_root}")
    return sorted([str(p) for p in pred_root.rglob("*.tif")])

top_level_tifs = find_top_level_tifs(DATA_DIR)

def find_chm_for_area(area):
    """Multi-area CHM discovery via template."""
    tmpl = CONFIG.get("CHM_TEMPLATE", "{area}_CHM.tif")
    target = tmpl.format(area=area).lower()
    for f in top_level_tifs:
        if os.path.basename(f).lower() == target:
            return f
    return None

def find_lidar_fp_for_area(area):
    tmpl = CONFIG.get("LIDAR_FOOTPRINT_TEMPLATE", "{area}_lidar_footprint.tif")
    target = tmpl.format(area=area).lower()
    for f in top_level_tifs:
        if os.path.basename(f).lower() == target:
            return f
    return None

# ----- Reference area (single-area path, and as "template" for multi) -----
# CHM (reference)
chm_file = None
for f in top_level_tifs:
    if os.path.basename(f).lower() == CONFIG["CHM_FILENAME"].lower():
        chm_file = f; break

predictor_files = find_predictors_in_subdir(PRED_DIR)

assert chm_file is not None, "Reference CHM not found"
assert len(predictor_files) > 0, "No predictor layers found in reference subdir"

with rasterio.open(chm_file) as ref:
    ref_profile = ref.profile
    ref_transform = ref.transform
    height, width = ref.height, ref.width
    ref_crs = ref.crs
    print(f"Reference area grid: {width}x{height} | CRS={ref_crs}")

## Read labels and reference-area masks

In [ ]:
# -------------------------
# 5) Labels / helpers (reference area)
# -------------------------
def read_single_band(path):
    with rasterio.open(path) as ds:
        return ds.read(1)

print("Reading the CHM for the reference area...")
chm = read_single_band(chm_file)
forest_label = (np.where(np.isfinite(chm), chm, 0) > 0).astype(np.uint8)  # ORIGINAL semantics

pos_frac = float(np.mean(forest_label))
neg_frac = 1.0 - pos_frac
print(f"Ref class balance - forest={pos_frac:.3f}, non-forest={neg_frac:.3f}")

def coords_from_rc(transform, rc):
    xs = transform.c + rc[:,1]*transform.a + rc[:,0]*transform.b
    ys = transform.f + rc[:,1]*transform.d + rc[:,0]*transform.e
    return list(zip(xs, ys))

def sample_band_values(path, coords_list):
    with rasterio.open(path) as ds:
        vals = np.array([v[0] for v in ds.sample(coords_list)])
    vals = np.where(np.isfinite(vals), vals, np.nan)
    return vals

lidar_ref_file = find_lidar_fp_for_area(AREA_TOKEN)
if lidar_ref_file is not None:
    lidar_ref_arr = read_single_band(lidar_ref_file)

    # Treat 1 as "LiDAR present", 255 as "outside / nodata"
    lidar_ref_mask = np.zeros_like(lidar_ref_arr, dtype=np.uint8)
    lidar_ref_mask[np.isfinite(lidar_ref_arr) & (lidar_ref_arr == 1)] = 1

    print(f"LiDAR footprint found for reference area {AREA_TOKEN}")
    print("   LiDAR mask stats:",
          "mean=", float(lidar_ref_mask.mean()),
          " #1s=", int(lidar_ref_mask.sum()),
          " #0s=", int(lidar_ref_mask.size - lidar_ref_mask.sum()))
else:
    lidar_ref_mask = None
    print(f"No LiDAR footprint found for reference area {AREA_TOKEN}.")

## Select and validate predictor channels

In [ ]:
# -------------------------
# 6) Feature selection
#    Priority:
#      (1) RGB_ONLY override
#      (2) reuse saved JSON for this AREA_CODE (filename-tagged)
#      (3) correlation pruning (build + save JSON for this AREA_CODE)
#      (4) fallback to all predictors
# -------------------------

selected_files = None

AREA_CODE = str(CONFIG.get("AREA_TOKEN", "")).strip()
if not AREA_CODE:
    raise ValueError('CONFIG["AREA_CODE"] must be set (e.g., "N22").')

# Base JSON name from config, but we will suffix with _{AREA_CODE} before .json
base_json_name = CONFIG["SAVE_SELECTED_FEATURES_JSON"]  # e.g. "selected_features.json"
base_root, base_ext = os.path.splitext(base_json_name)
if base_ext.lower() != ".json":
    base_ext = ".json"

sel_json_filename = f"{base_root}_{AREA_CODE}{base_ext}"   # e.g. "selected_features_N22.json"
sel_json_path = os.path.join(CONFIG["OUT_DIR"], sel_json_filename)

def _pick_rgb_files(all_predictor_paths, rgb_basenames):
    """Return predictor paths matching requested RGB basenames (case-insensitive)."""
    want = [b.lower() for b in rgb_basenames]
    by_base_lower = {os.path.basename(p).lower(): p for p in all_predictor_paths}
    picked = [by_base_lower[b] for b in want if b in by_base_lower]
    missing = [b for b in want if b not in by_base_lower]
    return picked, missing

def _load_selected_from_json(json_path, all_predictor_paths):
    """
    Load selection JSON and map entries back to actual predictor paths.

    Supports JSON entries that are either:
      - full paths (we use basename)
      - basenames
    """
    with open(json_path, "r") as f:
        pre = json.load(f)

    entries = pre.get("selected_predictors", [])
    if not isinstance(entries, list):
        entries = []

    wanted_basenames = [os.path.basename(str(x)) for x in entries]
    by_base = {os.path.basename(p): p for p in all_predictor_paths}

    selected = [by_base[b] for b in wanted_basenames if b in by_base]
    missing = [b for b in wanted_basenames if b not in by_base]
    return selected, missing

# --- RGB-only override ---
if bool(CONFIG.get("RGB_ONLY", False)):
    rgb_names = CONFIG.get("RGB_BASENAMES", ["red.tif", "green.tif", "blue.tif"])
    rgb_files, missing = _pick_rgb_files(predictor_files, rgb_names)

    if missing:
        print("RGB_ONLY is enabled but these files were not found in predictor subdir:")
        for m in missing:
            print("   -", m)
        raise FileNotFoundError(
            "RGB_ONLY enabled but one or more RGB band TIFFs are missing. "
            f"Expected: {rgb_names}"
        )

    selected_files = list(rgb_files)
    print(f"RGB_ONLY enabled - using RGB predictors: {[os.path.basename(p) for p in selected_files]}")

else:
    # --- (2) Reuse JSON selection if available FOR THIS AREA_CODE ---
    if os.path.exists(sel_json_path):
        try:
            selected_files, missing = _load_selected_from_json(sel_json_path, predictor_files)

            if missing:
                print(f"Predictors listed in {os.path.basename(sel_json_path)} but not found in reference area:")
                for m in missing:
                    print("   -", m)

            if selected_files:
                print(f"Reusing selected features JSON for AREA_CODE={AREA_CODE}: {os.path.basename(sel_json_path)}")
        except Exception as e:
            print("Failed to reuse selection JSON, falling back to pruning/all predictors:", e)
            selected_files = None

    # --- (3) Correlation pruning if JSON missing/empty for this area ---
    if selected_files is None or len(selected_files) == 0:
        do_prune = bool(CONFIG.get("DO_CORR_PRUNING", True))
        if do_prune and len(predictor_files) > 1:
            print(f"Sampling for correlation pruning (AREA_CODE={AREA_CODE})...")
            print(f"   (Will save to {os.path.basename(sel_json_path)})")

            rng = np.random.default_rng(int(CONFIG.get("CORR_RNG_SEED", 42)))
            step_r = max(1, height // int(CONFIG.get("CORR_GRID_DIV", 512)))
            step_c = max(1, width  // int(CONFIG.get("CORR_GRID_DIV", 512)))

            coords_all = np.array(
                [(r, c) for r in range(0, height, step_r) for c in range(0, width, step_c)],
                dtype=np.int32
            )
            rng.shuffle(coords_all)

            n_samp = int(CONFIG.get("NUM_SAMPLES_FOR_CORR", 5000))
            coords_all = coords_all[:min(len(coords_all), n_samp)]

            # xy should match what sample_band_values expects (same as your old script)
            xy = np.array(coords_from_rc(ref_transform, coords_all), dtype=np.float64)

            vals = []
            for pth in tqdm(predictor_files, desc="Sample corr"):
                vals.append(sample_band_values(pth, xy))
            vals = np.stack(vals, axis=1)  # (N, C)

            # Valid channels only (finite + non-constant)
            good_cols = []
            for j in range(vals.shape[1]):
                col = vals[:, j]
                if np.all(~np.isfinite(col)):
                    continue
                if np.nanstd(col) == 0:
                    continue
                good_cols.append(j)

            if len(good_cols) == 0:
                print("Correlation pruning found 0 valid channels; using all predictors.")
                selected_files = list(predictor_files)
            else:
                vals = vals[:, good_cols]
                pruned_files = [predictor_files[j] for j in good_cols]
                print(f"   Valid channels: {vals.shape[1]}")

                A = np.nan_to_num(vals, nan=np.nanmean(vals))
                corr = np.abs(np.corrcoef(A, rowvar=False))
                np.fill_diagonal(corr, 0.0)

                keep = np.ones(corr.shape[0], dtype=bool)
                th = float(CONFIG.get("CORR_ABS_THRESHOLD", 0.95))

                for i in range(corr.shape[0]):
                    if not keep[i]:
                        continue
                    for j in range(i + 1, corr.shape[1]):
                        if keep[j] and corr[i, j] > th:
                            keep[j] = False

                selected_files = [f for k, f in enumerate(pruned_files) if keep[k]]
                print(f"Selected {len(selected_files)} predictors (|r|<={th}).")

                # Save selection JSON (store basenames for portability)
                try:
                    os.makedirs(os.path.dirname(sel_json_path), exist_ok=True)
                    with open(sel_json_path, "w") as f:
                        json.dump(
                            {"selected_predictors": [os.path.basename(p) for p in selected_files]},
                            f,
                            indent=2
                        )
                    print(f"Saved selected features JSON: {os.path.basename(sel_json_path)}")
                except Exception as e:
                    print("Could not save selected features JSON:", e)
        else:
            print("Correlation pruning disabled or not enough predictors; using all predictors.")
            selected_files = list(predictor_files)


# --- Multi-area predictor harmonization (intersection of per-area selected_features_*.json) ---
# If you have per-area selections saved (e.g., selected_features_N22.json, selected_features_AE25.json, ...),
# this step can build a single "common" subset containing only predictors present in *all* available areas.
# This helps ensure a consistent channel set across areas and avoids training on predictors missing in some areas.
if USE_MULTI and (not bool(CONFIG.get("RGB_ONLY", False))) and bool(CONFIG.get("MULTI_USE_COMMON_SELECTED_FEATURES", True)):
    try:
        # Use the same base filename pattern as the per-area JSONs
        _base_json_name = CONFIG["SAVE_SELECTED_FEATURES_JSON"]  # e.g. "selected_features.json"
        _base_root, _base_ext = os.path.splitext(_base_json_name)
        if _base_ext.lower() != ".json":
            _base_ext = ".json"

        _area_sets = []
        _areas_used = []
        _missing_json = []

        for _a in CONFIG.get("AREA_CODES", []):
            _jp = os.path.join(CONFIG["OUT_DIR"], f"{_base_root}_{_a}{_base_ext}")
            if not os.path.exists(_jp):
                _missing_json.append(os.path.basename(_jp))
                continue
            try:
                with open(_jp, "r") as _f:
                    _data = json.load(_f)
                _ents = _data.get("selected_predictors", [])
                if not isinstance(_ents, list) or len(_ents) == 0:
                    continue
                _ents = [os.path.basename(str(x)) for x in _ents]
                _area_sets.append(set(_ents))
                _areas_used.append(_a)
            except Exception as _e:
                print(f"Could not read {_jp}: {_e}")

        if len(_area_sets) >= 2:
            _common = set.intersection(*_area_sets)
        elif len(_area_sets) == 1:
            _common = set(_area_sets[0])
        else:
            _common = set()

        if _common:
            _common_sorted = sorted(_common)

            _out_common = os.path.join(CONFIG["OUT_DIR"], f"{_base_root}_ALL_COMMON{_base_ext}")
            try:
                with open(_out_common, "w") as _f:
                    json.dump(
                        {
                            "selected_predictors": _common_sorted,
                            "mode": "intersection",
                            "areas_used": _areas_used,
                            "missing_json_files": _missing_json,
                        },
                        _f,
                        indent=2,
                    )
                print(f"Built common predictor subset across areas ({len(_common_sorted)} predictors): {os.path.basename(_out_common)}")
            except Exception as _e:
                print(f"Failed to write common predictor JSON: {_e}")

            # Override selected_files to use the common subset (mapped onto the reference-area predictor paths)
            _by_base_lower = {os.path.basename(p).lower(): p for p in predictor_files}
            _sel = []
            _miss = []
            for _b in _common_sorted:
                _k = str(_b).lower()
                if _k in _by_base_lower:
                    _sel.append(_by_base_lower[_k])
                else:
                    _miss.append(_b)

            if _miss:
                print("Common predictors missing in reference predictor folder (will be ignored):", _miss)

            if len(_sel) > 0:
                selected_files = _sel
                print(f"Using common multi-area predictor set (n={len(selected_files)})")
            else:
                print("Common predictor set mapped to 0 predictors in reference folder; keeping prior selection.")
        else:
            print("No common predictors found across available per-area JSONs; keeping prior selection.")
    except Exception as _e:
        print("Common predictor harmonization failed; keeping prior selection:", _e)


# --- Optional local cache for speed (area-specific to avoid basename collisions) ---
def ensure_local_cache(paths, cache_dir, area_token=None):
    """
    Copy predictor paths into an area-specific cache folder so different
    areas with identical basenames don't clobber each other.
    """
    if area_token is not None:
        cache_dir = os.path.join(cache_dir, str(area_token))

    os.makedirs(cache_dir, exist_ok=True)
    out = []

    for p in paths:
        base = os.path.basename(p)
        tgt = os.path.join(cache_dir, base)

        if not os.path.exists(tgt):
            try:
                shutil.copy2(p, tgt)
            except Exception as e:
                print("Copy failed:", e)
                tgt = p

        out.append(tgt)

    return out

def quick_validate_tif(path, H, W, num_windows=3, patch=64):
    with rasterio.open(path) as ds:
        if ds.height != H or ds.width != W:
            raise ValueError(f"Shape mismatch for {os.path.basename(path)}")
        for _ in range(num_windows):
            r = np.random.randint(0, max(1, H - patch))
            c = np.random.randint(0, max(1, W - patch))
            _ = ds.read(1, window=Window(c, r, patch, patch))
    return True

# Cache (still works for RGB-only too)
if CONFIG.get("USE_LOCAL_CACHE", True):
    selected_files = ensure_local_cache(selected_files, CONFIG["LOCAL_CACHE_DIR"], AREA_TOKEN)

# Validate
good = []
for f in selected_files:
    try:
        quick_validate_tif(f, height, width, patch=CONFIG["PATCH_SIZE"] // 2)
        good.append(f)
    except Exception as e:
        print(f"Dropping {os.path.basename(f)} due to:", e)

selected_files = good
assert len(selected_files) > 0, "No valid predictors remain in reference area."

selected_basenames = [os.path.basename(p) for p in selected_files]
print(f"Using {len(selected_files)} predictors in reference area: {selected_basenames}")

## Calculate normalization statistics

In [ ]:
# -------------------------
# 7) Normalization stats
#    - single-area: like original
#    - multi-area: global across all areas, saved to JSON
# -------------------------
PATCH = int(CONFIG["PATCH_SIZE"])
STRIDE = int(CONFIG["PATCH_STRIDE"])

def inclusive_starts(L, patch, stride):
    starts = list(range(0, max(1, L - patch + 1), max(1, stride)))
    if not starts or starts[-1] != L - patch:
        starts.append(max(0, L - patch))
    return starts

rows_ref = inclusive_starts(height, PATCH, STRIDE)
cols_ref = inclusive_starts(width, PATCH, STRIDE)
total_windows_ref = len(rows_ref) * len(cols_ref)

print("Opening reference predictors for cached reads...")
ds_paths = list(selected_files)
ds_list = [rasterio.open(p) for p in ds_paths]

def _safe_read_with_retry(idx, window):
    tries = int(CONFIG["READ_RETRIES"]) + 1
    for _ in range(tries):
        try:
            return ds_list[idx].read(1, window=window, boundless=False)
        except Exception:
            try:
                ds_list[idx].close()
            except:
                pass
            ds_list[idx] = rasterio.open(ds_paths[idx])
    return np.zeros((window.height, window.width), dtype=np.float32)

def stack_window_cached(window):
    bands = []
    for i in range(len(ds_list)):
        arr = _safe_read_with_retry(i, window)
        arr = np.where(np.isfinite(arr), arr, 0).astype(np.float32)
        bands.append(arr)
    return np.stack(bands, axis=-1)

norm_json_path = os.path.join(CONFIG["OUT_DIR"], CONFIG["SAVE_NORM_STATS_JSON"])
CHANNEL_MEAN = None
CHANNEL_STD  = None

# Try to reuse existing norm stats (for multi-area, global stats)
RECOMP_NORM = bool(CONFIG.get("RECOMPUTE_NORM_STATS_EACH_RUN", False))

if USE_MULTI and (not RECOMP_NORM) and os.path.exists(norm_json_path):
    try:
        with open(norm_json_path, "r") as f:
            norm_data = json.load(f)
        CHANNEL_MEAN = np.array(norm_data["mean"], dtype=np.float32)
        CHANNEL_STD  = np.array(norm_data["std"],  dtype=np.float32)
        print(f"Loaded GLOBAL normalization stats from {norm_json_path}")
    except Exception as e:
        print("Failed to load normalization stats JSON, recomputing:", e)
        CHANNEL_MEAN = None
        CHANNEL_STD  = None
else:
    if USE_MULTI and RECOMP_NORM:
        print("RECOMPUTE_NORM_STATS_EACH_RUN=True - recomputing GLOBAL norm stats (ignoring saved JSON).")

def compute_norm_stats_single():
    """Original single-area normalization stats (reference area only)."""
    rng = np.random.default_rng(1234)
    samples = []
    want = int(CONFIG["NUM_SAMPLES_FOR_NORM"])
    taken = 0
    for r in rows_ref:
        for c in cols_ref:
            w = Window(c, r, PATCH, PATCH)
            X = stack_window_cached(w).astype(np.float32)
            flat = X.reshape(-1, X.shape[-1])
            take = min(2048, flat.shape[0])
            idx = rng.choice(flat.shape[0], size=take, replace=False)
            samples.append(flat[idx]); taken += take
            if taken >= want:
                S = np.concatenate(samples, 0)
                return np.mean(S, 0).astype(np.float32), (np.std(S, 0)+1e-6).astype(np.float32)
    S = np.concatenate(samples, 0)
    return np.mean(S, 0).astype(np.float32), (np.std(S, 0)+1e-6).astype(np.float32)

def compute_norm_stats_global():
    """
    Global normalization across all areas in CONFIG["AREA_CODES"].
    Uses the same predictor set (basenames) as the reference area.
    """
    def sample_area(area_code, per_area_samples):
        print(f"\n Sampling predictors for normalization in area {area_code}")
        # Predictors for this area matching selected_basenames
        pred_subdir_a = CONFIG.get("PREDICTOR_SUBDIR_TEMPLATE", "{area}_layers").format(area=area_code)
        pred_dir_a = DATA_DIR / pred_subdir_a
        try:
            predictors_a = find_predictors_in_subdir(pred_dir_a)
        except FileNotFoundError as e:
            print(f"   {e}, skipping area.")
            return None

        by_base_a = {os.path.basename(p): p for p in predictors_a}
        missing_a = [b for b in selected_basenames if b not in by_base_a]
        if missing_a:
            print(f"   Area {area_code} missing some predictors; skipping for normalization.")
            for m in missing_a:
                print("      -", m)
            return None

        selected_paths_a = [by_base_a[b] for b in selected_basenames]

        # inside sample_area(area_code, per_area_samples):
        if CONFIG.get("USE_LOCAL_CACHE", True):
            selected_paths_a = ensure_local_cache(selected_paths_a,
                                                  CONFIG["LOCAL_CACHE_DIR"],
                                                  area_token=area_code)

        ds_paths_a = list(selected_paths_a)
        ds_list_a = [rasterio.open(p) for p in ds_paths_a]

        # For grid size, use first predictor shape
        with rasterio.open(ds_paths_a[0]) as ds0:
            H_a, W_a = ds0.height, ds0.width

        rows_a = inclusive_starts(H_a, PATCH, STRIDE)
        cols_a = inclusive_starts(W_a, PATCH, STRIDE)

        def _safe_read_with_retry_a(idx, window):
            tries = int(CONFIG["READ_RETRIES"]) + 1
            for _ in range(tries):
                try:
                    return ds_list_a[idx].read(1, window=window, boundless=False)
                except Exception:
                    try:
                        ds_list_a[idx].close()
                    except:
                        pass
                    ds_list_a[idx] = rasterio.open(ds_paths_a[idx])
            return np.zeros((window.height, window.width), dtype=np.float32)

        def stack_window_cached_a(window):
            bands = []
            for i in range(len(ds_list_a)):
                arr = _safe_read_with_retry_a(i, window)
                arr = np.where(np.isfinite(arr), arr, 0).astype(np.float32)
                bands.append(arr)
            return np.stack(bands, axis=-1)

        rng = np.random.default_rng(1234)
        samples = []
        taken = 0
        for r in rows_a:
            for c in cols_a:
                w = Window(c, r, PATCH, PATCH)
                X = stack_window_cached_a(w).astype(np.float32)
                flat = X.reshape(-1, X.shape[-1])
                take = min(2048, flat.shape[0])
                idx = rng.choice(flat.shape[0], size=take, replace=False)
                samples.append(flat[idx]); taken += take
                if taken >= per_area_samples:
                    break
            if taken >= per_area_samples:
                break

        # Close ds
        for ds_a in ds_list_a:
            try:
                ds_a.close()
            except:
                pass

        if not samples:
            print(f"   No samples collected for area {area_code}")
            return None

        S_area = np.concatenate(samples, axis=0)
        print(f"   Collected {S_area.shape[0]} samples for {area_code}")
        return S_area

    per_area = int(CONFIG["NUM_SAMPLES_FOR_NORM"])
    all_samples = []
    for area_code in CONFIG["AREA_CODES"]:
        S_area = sample_area(area_code, per_area)
        if S_area is not None:
            all_samples.append(S_area)

    if not all_samples:
        raise RuntimeError("Could not collect normalization samples from any area.")

    S_global = np.concatenate(all_samples, axis=0)
    print(f"\n Global normalization sample array shape: {S_global.shape}")
    mean = np.mean(S_global, axis=0).astype(np.float32)
    std  = (np.std(S_global, axis=0) + 1e-6).astype(np.float32)

    # Save JSON
    payload = {
        "mean": mean.tolist(),
        "std":  std.tolist(),
        "num_samples_per_area": per_area,
        "areas_used": CONFIG["AREA_CODES"],
    }
    with open(norm_json_path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved GLOBAL normalization stats JSON to {norm_json_path}")
    return mean, std

if USE_MULTI:
    if CHANNEL_MEAN is None or CHANNEL_STD is None:
        CHANNEL_MEAN, CHANNEL_STD = compute_norm_stats_global()
else:
    CHANNEL_MEAN, CHANNEL_STD = compute_norm_stats_single()

print("Normalization statistics are ready. The notebook will now identify and prepare training patches.")
print("   CHANNEL_MEAN shape:", CHANNEL_MEAN.shape)
print("   CHANNEL_STD  shape:", CHANNEL_STD.shape)

## Mine and materialize training, validation, and test patches

In [ ]:
# -------------------------
# 8) Patch mining (train/val) - precision-oriented
#    Single-area: original behaviour.
#    Multi-area : loop over areas, then materialise+concat.
#    LiDAR is applied *AFTER* patch selection as a post-filter.
# -------------------------
def iter_patch_windows_inclusive(H, W, patch, stride):
    for rr in inclusive_starts(H, patch, stride):
        for cc in inclusive_starts(W, patch, stride):
            yield Window(cc, rr, patch, patch), (rr, cc)

POS_THRESH   = float(CONFIG["POS_FRAC_THRESH"])
balance_posneg = bool(CONFIG.get("BALANCE_POS_NEG", True))
LIDAR_MIN_FRAC = float(CONFIG.get("LiDAR_MIN_COVER_FRAC", 0.99))

from scipy import ndimage as ndi

def _nan_guard(Xb, yb):
    if not np.isfinite(Xb).all() or not np.isfinite(yb).all():
        Xb = np.nan_to_num(Xb, nan=0.0, posinf=0.0, neginf=0.0)
        yb = np.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)
    return Xb, yb

def filter_by_lidar(coords_list, lidar_mask, min_frac, patch_size, label=""):
    """
    Post-hoc LiDAR gating: keep only patches with >= min_frac coverage of LiDAR mask.
    If lidar_mask is None, returns coords_list unchanged.

    coords_list entries: ((r, c), bool_label)
    label: optional string to identify which set (e.g. 'N22 pre-split')
    """
    if lidar_mask is None:
        return coords_list

    out = []
    dropped = 0
    for (r, c), lab in coords_list:
        lm = lidar_mask[r:r+patch_size, c:c+patch_size]
        if lm.size == 0:
            dropped += 1
            continue
        if float(lm.mean()) >= min_frac:
            out.append(((r, c), lab))
        else:
            dropped += 1

    if CONFIG.get("PRINT_LIDAR_STATS", True):
        tag = f" [{label}]" if label else ""
        print(f"   LiDAR post-filter{tag}: kept {len(out)} patches, dropped {dropped}")

    return out

def summarize_mix(name, coords_list, forest_label, PATCH):
    if not coords_list:
        print(f"   [{name}] No patches to summarise.")
        return
    fracs = []
    for (r, c), _lab in coords_list[:500]:  # sample up to 500
        patch = forest_label[r:r+PATCH, c:c+PATCH]
        fracs.append(float(patch.mean()))
    fracs = np.array(fracs)
    print(
        f"   [{name} SELECTED] approx forest_frac stats over patches "
        f"(sample n={len(fracs)}): mean={fracs.mean():.3f}, "
        f"p10={np.percentile(fracs,10):.3f}, p90={np.percentile(fracs,90):.3f}"
    )

# -------------------------------------------------
# Single-area mode: ORIGINAL behaviour + LiDAR post-filter
# -------------------------------------------------
if not USE_MULTI:
    print("Running in SINGLE-AREA mode.")
    # boundary ring
    lbl_bin = (forest_label.astype(np.uint8) > 0)
    er = ndi.binary_erosion(lbl_bin, iterations=1)
    dl = ndi.binary_dilation(lbl_bin, iterations=1)
    boundary = (dl ^ er).astype(np.uint8)

    rows = inclusive_starts(height, PATCH, STRIDE)
    cols = inclusive_starts(width, PATCH, STRIDE)
    total_windows = len(rows) * len(cols)

    candidate_pos  = []
    candidate_neg  = []
    candidate_edge = []

    for wdw, (r, c) in tqdm(iter_patch_windows_inclusive(height, width, PATCH, STRIDE),
                            total=total_windows,
                            desc="Patch mining (single-area)"):
        if r + PATCH > height or c + PATCH > width:
            continue

        lbl = forest_label[r:r+PATCH, c:c+PATCH]

        forest_frac = float(lbl.mean())
        edge_frac   = float(boundary[r:r+PATCH, c:c+PATCH].mean())

        # 1) Positive (forest) patches - OVERRIDES boundary
        if forest_frac > POS_THRESH:
            candidate_pos.append(((r, c), True))
            continue

        # 2) Boundary patches (mixed / ambiguous)
        if edge_frac >= 0.03:
            candidate_edge.append(((r, c), (forest_frac > 0.5)))
            continue

        # 3) Negative patches
        candidate_neg.append(((r, c), False))

    random.shuffle(candidate_edge)
    random.shuffle(candidate_pos)
    random.shuffle(candidate_neg)

    N_total = (
        len(candidate_edge) +
        len(candidate_pos) +
        len(candidate_neg)
    )

    if N_total == 0:
        raise RuntimeError("No candidate patches found for training/validation.")

    EDGE_FRAC = 0.20  # less boundary, more signal
    edge_take = min(len(candidate_edge), int(EDGE_FRAC * N_total))

    rem = N_total - edge_take

    # We want positives to be ~20% of the remaining pool, negatives ~80%
    POS_FRAC_IN_REM = 0.20

    pos_cap = int(POS_FRAC_IN_REM * rem)
    pos_take = min(len(candidate_pos), pos_cap)

    neg_take_total = max(0, rem - pos_take)

    neg_take = min(len(candidate_neg), neg_take_total)

    selected = (
        candidate_edge[:edge_take] +
        candidate_pos[:pos_take] +
        candidate_neg[:neg_take]
    )
    random.shuffle(selected)

    # LiDAR gating BEFORE train/val split
    selected = filter_by_lidar(
        selected,
        lidar_ref_mask,           # or lidar_mask for the single area, whichever name you're using
        LIDAR_MIN_FRAC,
        PATCH,
        label=f"{AREA_TOKEN} pre-split"
    )

    if len(selected) == 0:
        raise RuntimeError(f"No patches remain after LiDAR filtering for {AREA_TOKEN}.")

    summarize_mix(f"{AREA_TOKEN} SELECTED", selected, forest_label, PATCH)

    # --- Train/Val/Test split (true held-out test) ---
    train_ratio = float(CONFIG.get("TRAIN_RATIO", 0.60))
    val_ratio   = float(CONFIG.get("VAL_RATIO", 0.20))
    test_ratio  = float(CONFIG.get("TEST_RATIO", 0.20))

    # Normalize ratios defensively
    s = train_ratio + val_ratio + test_ratio
    train_ratio, val_ratio, test_ratio = train_ratio / s, val_ratio / s, test_ratio / s

    n = len(selected)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train_list = selected[:n_train]
    val_list   = selected[n_train:n_train + n_val]
    test_list  = selected[n_train + n_val:]

    # --- LiDAR post-filter (after patch selection; defensive, since we already gated pre-split) ---
    train_list = filter_by_lidar(train_list, lidar_ref_mask, LIDAR_MIN_FRAC, PATCH, label=f"{AREA_TOKEN} train")
    val_list   = filter_by_lidar(val_list,   lidar_ref_mask, LIDAR_MIN_FRAC, PATCH, label=f"{AREA_TOKEN} val")
    test_list  = filter_by_lidar(test_list,  lidar_ref_mask, LIDAR_MIN_FRAC, PATCH, label=f"{AREA_TOKEN} test")

    if CONFIG["MAX_TRAIN_PATCHES"]:
        train_list = train_list[:int(CONFIG["MAX_TRAIN_PATCHES"])]
    if CONFIG["MAX_VAL_PATCHES"]:
        val_list   = val_list[:int(CONFIG["MAX_VAL_PATCHES"])]
    if CONFIG.get("MAX_TEST_PATCHES", None):
        test_list  = test_list[:int(CONFIG["MAX_TEST_PATCHES"])]

    print(f"Train patches: {len(train_list)} | Val patches: {len(val_list)} | Test patches: {len(test_list)}")
    print(
        f"   Breakdown (pre-sampling): "
        f"edge={len(candidate_edge)}, pos={len(candidate_pos)}, "
        f"neg={len(candidate_neg)}"
    )

    # -------------------------
    # 9) MATERIALISE patches to /content (no raster I/O during fit)
    # -------------------------
    def materialize_patches(coords_list, out_prefix, max_items=None):
        """Writes X.npy and y.npy under /content/patch_cache; returns file paths."""
        os.makedirs("/content/patch_cache", exist_ok=True)
        Xs, Ys = [], []
        limit = len(coords_list) if max_items is None else min(len(coords_list), max_items)
        for i in range(limit):
            (r, c), _is_pos = coords_list[i]
            w = Window(c, r, PATCH, PATCH)
            Xw = stack_window_cached(w).astype(np.float32)
            Xw = (Xw - CHANNEL_MEAN) / CHANNEL_STD
            yw = forest_label[r:r+PATCH, c:c+PATCH].astype(np.float32)[..., None]
            Xw, yw = _nan_guard(Xw, yw)
            Xs.append(Xw); Ys.append(yw)
        X = np.stack(Xs, 0); y = np.stack(Ys, 0)
        x_path = f"/content/patch_cache/{out_prefix}_X.npy"
        y_path = f"/content/patch_cache/{out_prefix}_y.npy"
        np.save(x_path, X); np.save(y_path, y)
        return x_path, y_path

    print("Materialising train/val/test patches to /content ...")
    train_X_path, train_y_path = materialize_patches(train_list, "train")
    val_X_path,   val_y_path   = (materialize_patches(val_list,  "val")  if len(val_list)  > 0 else (None, None))
    test_X_path,  test_y_path  = (materialize_patches(test_list, "test") if len(test_list) > 0 else (None, None))
    print("Single-area training, validation, and test patch arrays have been saved in the local Colab cache.")

# -------------------------------------------------
# Multi-area mode: per-area patch mining + LiDAR post-filter,
# then concatenation into single train/val arrays.
# -------------------------------------------------
else:
    print("Running in MULTI-AREA mode.")

    global_train_cap = int(CONFIG["MAX_TRAIN_PATCHES"]) if CONFIG["MAX_TRAIN_PATCHES"] else 0
    global_val_cap   = int(CONFIG["MAX_VAL_PATCHES"])   if CONFIG["MAX_VAL_PATCHES"]   else 0
    # Optional test cap: default to MAX_VAL_PATCHES if MAX_TEST_PATCHES not provided
    global_test_cap  = int(CONFIG.get("MAX_TEST_PATCHES", CONFIG.get("MAX_VAL_PATCHES", 0))) if CONFIG.get("MAX_TEST_PATCHES", CONFIG.get("MAX_VAL_PATCHES", 0)) else 0

    # initialise chunk accumulators as empty lists
    train_X_chunks = []
    train_y_chunks = []
    val_X_chunks   = []
    val_y_chunks   = []
    test_X_chunks  = []
    test_y_chunks  = []

    train_added = 0
    val_added   = 0
    test_added  = 0

    for area in CONFIG["AREA_CODES"]:
        print(f"\n=============================")
        print(f"Processing training data for area: {area}")
        print(f"=============================")

        # CHM and LiDAR footprint
        chm_a_path = find_chm_for_area(area)
        lidar_a_path = find_lidar_fp_for_area(area)

        if chm_a_path is None:
            print(f"   No CHM found for {area}, skipping.")
            continue
        if lidar_a_path is None:
            print(f"   No LiDAR footprint for {area}, skipping (LiDAR mandatory in multi-area).")
            continue

        chm_a = read_single_band(chm_a_path)
        forest_a = (np.where(np.isfinite(chm_a), chm_a, 0) > 0).astype(np.uint8)

        lidar_a_arr = read_single_band(lidar_a_path)
        lidar_a = np.zeros_like(lidar_a_arr, dtype=np.uint8)
        lidar_a[np.isfinite(lidar_a_arr) & (lidar_a_arr == 1)] = 1

        H_a, W_a = forest_a.shape
        forest_frac_a = float(forest_a.mean())
        print(f"   Size={W_a}x{H_a}, forest_frac={forest_frac_a:.3f}")

        # Predictors for this area matching the reference basenames
        pred_subdir_a = CONFIG.get("PREDICTOR_SUBDIR_TEMPLATE", "{area}_layers").format(area=area)
        pred_dir_a = DATA_DIR / pred_subdir_a
        try:
            predictors_a = find_predictors_in_subdir(pred_dir_a)
        except FileNotFoundError as e:
            print(f"   {e}, skipping area.")
            continue

        by_base_a = {os.path.basename(p): p for p in predictors_a}
        missing_a = [b for b in selected_basenames if b not in by_base_a]
        if missing_a:
            print(f"   Area {area} missing required predictors; skipping.")
            for m in missing_a:
                print("      -", m)
            continue

        selected_a_paths = [by_base_a[b] for b in selected_basenames]

        if CONFIG.get("USE_LOCAL_CACHE", True):
            selected_a_paths = ensure_local_cache(selected_a_paths,
                                                  CONFIG["LOCAL_CACHE_DIR"],
                                                  area_token=area)
        # Shape checks
        valid_paths_a = []
        for p in selected_a_paths:
            try:
                quick_validate_tif(p, H_a, W_a, patch=PATCH//2)
                valid_paths_a.append(p)
            except Exception as e:
                print(f"   Dropping predictor {os.path.basename(p)} in area {area}: {e}")
        selected_a_paths = valid_paths_a
        if len(selected_a_paths) == 0:
            print(f"   No valid predictors for {area}; skipping.")
            continue

        ds_paths_a = list(selected_a_paths)
        ds_list_a = [rasterio.open(p) for p in ds_paths_a]

        def _safe_read_with_retry_a(idx, window):
            tries = int(CONFIG["READ_RETRIES"]) + 1
            for _ in range(tries):
                try:
                    return ds_list_a[idx].read(1, window=window, boundless=False)
                except Exception:
                    try:
                        ds_list_a[idx].close()
                    except:
                        pass
                    ds_list_a[idx] = rasterio.open(ds_paths_a[idx])
            return np.zeros((window.height, window.width), dtype=np.float32)

        def stack_window_cached_a(window):
            bands = []
            for i in range(len(ds_list_a)):
                arr = _safe_read_with_retry_a(i, window)
                arr = np.where(np.isfinite(arr), arr, 0).astype(np.float32)
                bands.append(arr)
            return np.stack(bands, axis=-1)

        # Boundary map
        lbl_bin_a = (forest_a.astype(np.uint8) > 0)
        er_a = ndi.binary_erosion(lbl_bin_a, iterations=1)
        dl_a = ndi.binary_dilation(lbl_bin_a, iterations=1)
        boundary_a = (dl_a ^ er_a).astype(np.uint8)

        rows_a = inclusive_starts(H_a, PATCH, STRIDE)
        cols_a = inclusive_starts(W_a, PATCH, STRIDE)
        total_windows_a = len(rows_a) * len(cols_a)

        candidate_pos_a  = []
        candidate_neg_a  = []
        candidate_edge_a = []

        for wdw, (r, c) in tqdm(
            iter_patch_windows_inclusive(H_a, W_a, PATCH, STRIDE),
            total=total_windows_a,
            desc=f"Patch mining {area}"
        ):
            if r + PATCH > H_a or c + PATCH > W_a:
                continue

            lbl = forest_a[r:r+PATCH, c:c+PATCH]
            edge = boundary_a[r:r+PATCH, c:c+PATCH]

            forest_frac = float(lbl.mean())
            edge_frac   = float(edge.mean())

            # 1) Positive
            if forest_frac > POS_THRESH:
                candidate_pos_a.append(((r, c), True))
                continue

            # 2) Boundary
            if edge_frac >= 0.03:
                candidate_edge_a.append(((r, c), (forest_frac > 0.5)))
                continue

            candidate_neg_a.append(((r, c), False))

        random.shuffle(candidate_edge_a)
        random.shuffle(candidate_pos_a)
        random.shuffle(candidate_neg_a)

        N_total_a = (
            len(candidate_edge_a) +
            len(candidate_pos_a) +
            len(candidate_neg_a)
        )
        if N_total_a == 0:
            print(f"   [{area}] No candidate patches, skipping.")
            for ds_a in ds_list_a:
                try: ds_a.close()
                except: pass
            continue

        EDGE_FRAC = 0.20  # less boundary, more signal
        edge_take_a = min(len(candidate_edge_a), int(EDGE_FRAC * N_total_a))

        rem = N_total_a - edge_take_a

        # We want positives to be ~20% of the remaining pool, negatives ~80%
        POS_FRAC_IN_REM = 0.20

        pos_cap_a = int(POS_FRAC_IN_REM * rem)
        pos_take_a = min(len(candidate_pos_a), pos_cap_a)

        neg_take_total_a = max(0, rem - pos_take_a)

        neg_take_a = min(len(candidate_neg_a), neg_take_total_a)

        selected_a = (
            candidate_edge_a[:edge_take_a] +
            candidate_pos_a[:pos_take_a] +
            candidate_neg_a[:neg_take_a]
        )
        random.shuffle(selected_a)

        print(
            f"   [{area}] selected (pre-LiDAR): "
            f"edge={min(len(candidate_edge_a), edge_take_a)}, "
            f"pos={min(len(candidate_pos_a), pos_take_a)}, "
            f"neg={min(len(candidate_neg_a), neg_take_a)}"
        )

        # LiDAR gating for this area BEFORE its train/val split
        selected_a = filter_by_lidar(
            selected_a,
            lidar_a,
            LIDAR_MIN_FRAC,
            PATCH,
            label=f"{area} pre-split"
        )
        print(f"   [{area}] selected AFTER LiDAR: {len(selected_a)} patches total")

        if len(selected_a) == 0:
            print(f"   [{area}] No patches remain after LiDAR filtering; skipping area.")
            # close ds_list_a etc. as you already do, then continue
            for ds_a in ds_list_a:
                try:
                    ds_a.close()
                except:
                    pass
            import gc; gc.collect()
            continue

        summarize_mix(f"{area} SELECTED", selected_a, forest_a, PATCH)

        # --- Train/Val/Test split (true held-out test) ---
        train_ratio_a = float(CONFIG.get("TRAIN_RATIO", 0.60))
        val_ratio_a   = float(CONFIG.get("VAL_RATIO", 0.20))
        test_ratio_a  = float(CONFIG.get("TEST_RATIO", 0.20))

        s_a = train_ratio_a + val_ratio_a + test_ratio_a
        train_ratio_a, val_ratio_a, test_ratio_a = train_ratio_a / s_a, val_ratio_a / s_a, test_ratio_a / s_a

        n_a = len(selected_a)
        n_train_a = int(n_a * train_ratio_a)
        n_val_a   = int(n_a * val_ratio_a)

        train_list_a = selected_a[:n_train_a]
        val_list_a   = selected_a[n_train_a:n_train_a + n_val_a]
        test_list_a  = selected_a[n_train_a + n_val_a:]

        print(
            f"   [{area}] train/val/test sizes after LiDAR & split: "
            f"train={len(train_list_a)}, val={len(val_list_a)}, test={len(test_list_a)}"
        )

        # Respect global caps
        if global_train_cap > 0:
            remaining_tr = max(0, global_train_cap - train_added)
            train_list_a = train_list_a[:remaining_tr]
        if global_val_cap > 0:
            remaining_val = max(0, global_val_cap - val_added)
            val_list_a    = val_list_a[:remaining_val]
        if global_test_cap > 0:
            remaining_te = max(0, global_test_cap - test_added)
            test_list_a  = test_list_a[:remaining_te]

        print(f"   [{area}] Train patches (after LiDAR & caps): {len(train_list_a)} | Val patches: {len(val_list_a)} | Test patches: {len(test_list_a)}")

        def summarize_patch_label_stats(name, coords_list, forest_mask, max_patches=200):
            if not coords_list:
                print(f"   [{name}] No patches to summarise.")
                return
            fracs = []
            for (r, c), _lab in coords_list[:max_patches]:
                lbl = forest_mask[r:r+PATCH, c:c+PATCH]
                fracs.append(float(lbl.mean()))
            fracs = np.array(fracs)
            print(
                f"   [{name}] patch label stats (first {len(fracs)}): "
                f"mean_frac={fracs.mean():.3f}, "
                f"min={fracs.min():.3f}, "
                f"max={fracs.max():.3f}"
            )

        summarize_patch_label_stats(f"{area} TRAIN", train_list_a, forest_a)
        summarize_patch_label_stats(f"{area} VAL",   val_list_a,   forest_a)
        summarize_patch_label_stats(f"{area} TEST",  test_list_a,  forest_a)

        # Materialise this area's patches into memory
        def materialize_area(coords_list):
            if not coords_list:
                return None, None
            Xs, Ys = [], []
            for (r, c), _is_pos in coords_list:
                w = Window(c, r, PATCH, PATCH)
                Xw = stack_window_cached_a(w).astype(np.float32)
                Xw = (Xw - CHANNEL_MEAN) / CHANNEL_STD
                yw = forest_a[r:r+PATCH, c:c+PATCH].astype(np.float32)[..., None]
                Xw, yw = _nan_guard(Xw, yw)
                Xs.append(Xw); Ys.append(yw)
            return np.stack(Xs, 0), np.stack(Ys, 0)

        X_tr_a, y_tr_a = materialize_area(train_list_a)
        X_val_a, y_val_a = materialize_area(val_list_a)
        X_te_a, y_te_a = materialize_area(test_list_a)

        if X_tr_a is not None:
            train_X_chunks.append(X_tr_a)
            train_y_chunks.append(y_tr_a)
            train_added += X_tr_a.shape[0]

        if X_val_a is not None:
            val_X_chunks.append(X_val_a)
            val_y_chunks.append(y_val_a)
            val_added += X_val_a.shape[0]
        if X_te_a is not None:
            test_X_chunks.append(X_te_a)
            test_y_chunks.append(y_te_a)
            test_added += X_te_a.shape[0]

        # Close predictors for this area
        for ds_a in ds_list_a:
            try: ds_a.close()
            except: pass
        import gc; gc.collect()

        # Stop early if we've filled caps
        if global_train_cap > 0 and train_added >= global_train_cap and \
           global_val_cap > 0 and val_added >= global_val_cap and \
           (global_test_cap == 0 or test_added >= global_test_cap):
            print("   Global train/val/test caps reached; stopping further areas.")
            break

    # Concatenate across all areas and write single train_X.npy / val_X.npy
    os.makedirs("/content/patch_cache", exist_ok=True)

    if train_X_chunks:
        train_X = np.concatenate(train_X_chunks, axis=0)
        train_y = np.concatenate(train_y_chunks, axis=0)
        train_X_path = "/content/patch_cache/train_X.npy"
        train_y_path = "/content/patch_cache/train_y.npy"
        np.save(train_X_path, train_X)
        np.save(train_y_path, train_y)
        print(f"\n Multi-area train set: {train_X.shape[0]} patches -> {train_X_path}, {train_y_path}")
        del train_X, train_y, train_X_chunks, train_y_chunks
    else:
        train_X_path = None
        train_y_path = None
        print("\n No multi-area train patches created.")

    if val_X_chunks:
        val_X = np.concatenate(val_X_chunks, axis=0)
        val_y = np.concatenate(val_y_chunks, axis=0)
        val_X_path = "/content/patch_cache/val_X.npy"
        val_y_path = "/content/patch_cache/val_y.npy"
        np.save(val_X_path, val_X)
        np.save(val_y_path, val_y)
        print(f"Multi-area val set: {val_X.shape[0]} patches -> {val_X_path}, {val_y_path}")
        del val_X, val_y, val_X_chunks, val_y_chunks
    else:
        val_X_path = None
        val_y_path = None
        print("No multi-area val patches created.")


    if test_X_chunks:
        test_X = np.concatenate(test_X_chunks, axis=0)
        test_y = np.concatenate(test_y_chunks, axis=0)
        test_X_path = "/content/patch_cache/test_X.npy"
        test_y_path = "/content/patch_cache/test_y.npy"
        np.save(test_X_path, test_X)
        np.save(test_y_path, test_y)
        print(f"Multi-area test set: {test_X.shape[0]} patches -> {test_X_path}, {test_y_path}")
        del test_X, test_y, test_X_chunks, test_y_chunks
    else:
        test_X_path = None
        test_y_path = None
        print("No multi-area test patches created.")
    print("Multi-area patch extraction & materialisation complete.")

## Create re-iterable training sequences and augmentation

In [ ]:
# =====================================================
# 10) Re-iterable Sequences (with augmentation)
# =====================================================
def _augment_pair(img, mask):
    """
    img:  (H,W,C) float32
    mask: (H,W) or (H,W,1) {0,1}
    returns augmented (img, mask) with shape (H,W,C) and (H,W,1)
    """
    img  = np.asarray(img, dtype=np.float32)
    mask = np.asarray(mask)
    if mask.ndim == 2:
        mask = mask[..., None]
    elif mask.shape[-1] > 1:
        mask = mask[..., :1]
    mask = mask.astype(np.float32)

    H, W, C = img.shape

    # 90 degrees  rotations
    k = np.random.randint(0, 4)
    if k > 0:
        img  = np.rot90(img,  k, axes=(0, 1))
        mask = np.rot90(mask, k, axes=(0, 1))

    # flips
    if np.random.rand() < 0.5:
        img  = np.flip(img,  axis=1)
        mask = np.flip(mask, axis=1)
    if np.random.rand() < 0.5:
        img  = np.flip(img,  axis=0)
        mask = np.flip(mask, axis=0)

    # mild random scale + translate
    if np.random.rand() < 0.5:
        scale = 1.0 + np.random.uniform(-0.08, 0.08)
        newH, newW = max(8, int(H * scale)), max(8, int(W * scale))

        img2 = tf.image.resize(img,  (newH, newW), method="bilinear")
        msk2 = tf.image.resize(mask, (newH, newW), method="nearest")

        img2 = tf.image.resize_with_pad(img2, H, W, method="bilinear")
        msk2 = tf.image.resize_with_pad(msk2, H, W, method="nearest")

        offy = int(np.random.randint(-3, 4))
        offx = int(np.random.randint(-3, 4))
        img  = tf.roll(img2, shift=[offy, offx], axis=[0, 1])
        mask = tf.roll(msk2, shift=[offy, offx], axis=[0, 1])

        img  = img.numpy().astype(np.float32)
        mask = mask.numpy().astype(np.float32)

    # channel gain/bias
    if np.random.rand() < 0.8:
        gain = 1.0 + np.random.uniform(-0.10, 0.10, size=(1, 1, C)).astype(np.float32)
        bias = np.random.uniform(-0.10, 0.10, size=(1, 1, C)).astype(np.float32)
        img = img * gain + bias

    # light Gaussian noise
    if np.random.rand() < 0.3:
        img = img + np.random.normal(0.0, 0.02, size=img.shape).astype(np.float32)

    mask = (mask > 0.5).astype(np.float32)
    return img, mask


class NpySequence(Sequence):
    """
    Single .npy pair loader (original behaviour).
    """
    def __init__(self, x_path, y_path, batch_size, shuffle=True, augment=True):
        self.X = np.load(x_path, mmap_mode="r")
        self.y = np.load(y_path, mmap_mode="r")
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.augment = bool(augment) and CONFIG.get("AUGMENT", True)
        self.indexes = np.arange(self.X.shape[0])
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(self.X.shape[0] / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, i):
        start = i * self.batch_size
        end   = min((i + 1) * self.batch_size, self.X.shape[0])
        sel   = self.indexes[start:end]
        Xb = self.X[sel].astype(np.float32)
        yb = self.y[sel].astype(np.float32)

        if self.augment:
            out_X, out_y = [], []
            for k in range(Xb.shape[0]):
                Xi, yi = _augment_pair(Xb[k], yb[k, ..., 0])
                out_X.append(Xi); out_y.append(yi[..., None])
            Xb = np.stack(out_X, 0); yb = np.stack(out_y, 0)

        Xb = np.nan_to_num(Xb, nan=0.0, posinf=0.0, neginf=0.0)
        yb = np.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)
        return Xb, yb


class MultiNpySequence(Sequence):
    """
    Multi-file sequence for multi-area training.
    Streams from several .npy files using mmap, without loading all into RAM.
    x_paths / y_paths are lists of .npy files; each pair (X[i], y[i]) is an area.
    """
    def __init__(self, x_paths, y_paths, batch_size, shuffle=True, augment=True):
        assert len(x_paths) == len(y_paths), "X/Y file lists must match length."
        self.x_paths = list(x_paths)
        self.y_paths = list(y_paths)
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.augment = bool(augment) and CONFIG.get("AUGMENT", True)

        # mmap all arrays (small metadata + file handles, not full load)
        self.X_arrays = [np.load(p, mmap_mode="r") for p in self.x_paths]
        self.y_arrays = [np.load(p, mmap_mode="r") for p in self.y_paths]

        # build a flat index: each entry = (file_id, local_idx)
        self.sample_index = []
        for fid, Xa in enumerate(self.X_arrays):
            n = Xa.shape[0]
            for i in range(n):
                self.sample_index.append((fid, i))

        self.indexes = np.arange(len(self.sample_index))
        self.on_epoch_end()

        # sanity check: channels consistent
        ch = self.X_arrays[0].shape[-1]
        for Xa in self.X_arrays[1:]:
            if Xa.shape[-1] != ch:
                raise ValueError("Inconsistent channel count across X files.")

    def __len__(self):
        return int(np.ceil(len(self.sample_index) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end   = min((idx + 1) * self.batch_size, len(self.sample_index))
        sel   = self.indexes[start:end]

        X_list, y_list = [], []
        for k in sel:
            fid, local_idx = self.sample_index[k]
            Xa = self.X_arrays[fid]
            ya = self.y_arrays[fid]
            X_list.append(Xa[local_idx])
            y_list.append(ya[local_idx])

        Xb = np.stack(X_list, 0).astype(np.float32)
        yb = np.stack(y_list, 0).astype(np.float32)

        if self.augment:
            out_X, out_y = [], []
            for k in range(Xb.shape[0]):
                Xi, yi = _augment_pair(Xb[k], yb[k, ..., 0])
                out_X.append(Xi); out_y.append(yi[..., None])
            Xb = np.stack(out_X, 0); yb = np.stack(out_y, 0)

        Xb = np.nan_to_num(Xb, nan=0.0, posinf=0.0, neginf=0.0)
        yb = np.nan_to_num(yb, nan=0.0, posinf=0.0, neginf=0.0)
        return Xb, yb


# ---- Build the appropriate Sequence(s) depending on single vs multi area ----
BATCH_SIZE = int(CONFIG["BATCH_SIZE"])

if USE_MULTI:
    # New multi-area layout: single concatenated arrays
    train_X_path = "/content/patch_cache/train_X.npy"
    train_y_path = "/content/patch_cache/train_y.npy"
    val_X_path   = "/content/patch_cache/val_X.npy"
    val_y_path   = "/content/patch_cache/val_y.npy"
    test_X_path  = "/content/patch_cache/test_X.npy"
    test_y_path  = "/content/patch_cache/test_y.npy"

    if not (os.path.exists(train_X_path) and os.path.exists(train_y_path)):
        raise FileNotFoundError(
            "USE_MULTI_AREA=True but /content/patch_cache/train_X.npy or train_y.npy "
            "is missing. Run the multi-area patch-mining cell first."
        )

    train_seq = NpySequence(train_X_path, train_y_path, BATCH_SIZE,
                            shuffle=True, augment=True)

    if os.path.exists(val_X_path) and os.path.exists(val_y_path):
        val_seq = NpySequence(val_X_path, val_y_path, BATCH_SIZE,
                              shuffle=False, augment=False)
    else:
        val_seq = None
        print("No multi-area val set found; continuing without validation.")

    if os.path.exists(test_X_path) and os.path.exists(test_y_path):
        test_seq = NpySequence(test_X_path, test_y_path, BATCH_SIZE,
                               shuffle=False, augment=False)
    else:
        test_seq = None
        print("No multi-area test set found; skipping final test evaluation.")
else:
    # Single-area behaviour
    train_seq = NpySequence(train_X_path, train_y_path, BATCH_SIZE,
                            shuffle=True, augment=True)
    val_seq   = (NpySequence(val_X_path, val_y_path, BATCH_SIZE,
                             shuffle=False, augment=False)
                 if (val_X_path and val_y_path) else None)
    test_seq  = (NpySequence(test_X_path, test_y_path, BATCH_SIZE,
                             shuffle=False, augment=False)
                 if (test_X_path and test_y_path) else None)

steps_per_epoch = len(train_seq)
val_steps       = len(val_seq) if val_seq is not None else 0
test_steps      = len(test_seq) if ("test_seq" in globals() and test_seq is not None) else 0
print(f"Steps/epoch: train={steps_per_epoch}, val={val_steps}, test={test_steps}")

## Build the selected segmentation architecture

In [ ]:
# =====================================================
# 11) Model (UNet / UNet3+ / Pretrained ResNet50 variants)
# =====================================================

# --- Normalization that works well with small batches ---
try:
    from tensorflow_addons.layers import GroupNormalization as _GroupNorm
    def _Norm():
        return _GroupNorm(groups=8, axis=-1)
    print("Using GroupNorm (8 groups).")
except Exception as _e:
    # Fallback: per-channel LayerNorm (closer to InstanceNorm than global)
    def _Norm():
        return layers.LayerNormalization(axis=-1)
    print("tensorflow-addons not found; using LayerNorm(axis=-1) fallback.")


def aspp(x, filters, reg, rates=(1, 6, 12, 18)):
    """
    Atrous Spatial Pyramid Pooling (static-size variant).
    Requires that x.shape[1] and x.shape[2] are known integers (your patch size).
    """
    H, W = x.shape[1], x.shape[2]
    if H is None or W is None:
        raise ValueError(
            "Static-size ASPP requires known spatial dims at build time; "
            "got H/W=None. Use a dynamic variant if shapes are dynamic."
        )
    H, W = int(H), int(W)

    branches = []

    # 1x1 branch
    b0 = layers.Conv2D(
        filters, 1, padding="same", kernel_regularizer=reg, activation="relu"
    )(x)
    branches.append(b0)

    # 3x3 atrous branches
    for r in rates[1:]:
        b = layers.Conv2D(
            filters, 3, padding="same", dilation_rate=r,
            kernel_regularizer=reg, activation="relu"
        )(x)
        branches.append(b)

    # Image-level features via global pooling -> 1x1 conv -> resize
    gp = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gp = layers.Conv2D(filters, 1, padding="same", kernel_regularizer=reg, activation="relu")(gp)
    gp = layers.Resizing(H, W, interpolation="bilinear")(gp)

    y = layers.Concatenate()(branches + [gp])
    y = layers.Conv2D(filters, 1, padding="same", kernel_regularizer=reg, activation="relu")(y)
    return y


# ---- Attention blocks ----
def cbam_block(x, ratio=8):
    ch = int(x.shape[-1])
    if ch is None:
        return x

    # Channel attention
    sc = layers.GlobalAveragePooling2D(keepdims=True)(x)
    sm = layers.GlobalMaxPooling2D(keepdims=True)(x)

    mlp = keras.Sequential([
        layers.Conv2D(max(ch // ratio, 1), 1, activation="relu"),
        layers.Conv2D(ch, 1, activation="sigmoid"),
    ])

    ch_att = layers.Add()([mlp(sc), mlp(sm)])
    ch_refined = layers.Multiply()([x, ch_att])

    # Spatial attention
    avg = tf.reduce_mean(ch_refined, axis=-1, keepdims=True)
    mx  = tf.reduce_max(ch_refined, axis=-1, keepdims=True)
    sp  = layers.Concatenate()([avg, mx])
    sp  = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(sp)
    out = layers.Multiply()([ch_refined, sp])
    return out


def squeeze_excite(y, r=8):
    if not CONFIG.get("USE_SE", True):
        return y
    ch = int(y.shape[-1]) if y.shape[-1] is not None else None
    if not ch:
        return y
    s = layers.GlobalAveragePooling2D(keepdims=True)(y)
    s = layers.Conv2D(max(ch // r, 1), 1, activation="relu")(s)
    s = layers.Conv2D(ch, 1, activation="sigmoid")(s)
    return layers.Multiply()([y, s])


def conv_block(x, filters, reg, pdrop=0.1):
    x = layers.Conv2D(filters, 3, padding="same", kernel_regularizer=reg)(x)
    x = _Norm()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filters, 3, padding="same", kernel_regularizer=reg)(x)
    x = _Norm()(x)
    x = layers.Activation("relu")(x)

    if CONFIG.get("USE_CBAM", False):
        x = cbam_block(x, ratio=int(CONFIG.get("CBAM_RATIO", 8)))
    else:
        x = squeeze_excite(x)

    x = layers.SpatialDropout2D(pdrop)(x)
    return x


def encoder_block(x, filters, reg):
    c = conv_block(x, filters, reg)
    p = layers.MaxPooling2D(2)(c)
    return c, p


def att_gate(x, g, filters):
    theta_x = layers.Conv2D(filters, 1, padding="same")(x)
    phi_g   = layers.Conv2D(filters, 1, padding="same")(g)
    a = layers.Activation("relu")(layers.Add()([theta_x, phi_g]))
    a = layers.Conv2D(1, 1, activation="sigmoid")(a)
    return layers.Multiply()([x, a])


def decoder_block(x, skip, filters, reg):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
    gated = att_gate(skip, x, filters)
    x = layers.Concatenate()([x, gated])
    x = conv_block(x, filters, reg)
    return x


def make_head(x, out_filters=1, name=None, out_size=None):
    """
    Create an output head and ensure the *last* op has the desired `name`,
    so model.output_names == [name, ...] and matches the compile loss dict.
    """
    y = layers.Conv2D(out_filters, 1, activation="sigmoid", dtype="float32", name=None)(x)
    if out_size is not None:
        y = layers.Resizing(out_size[0], out_size[1], interpolation="bilinear", name=name)(y)
    else:
        y = layers.Lambda(lambda t: t, name=name)(y)
    return y


def build_unet_with_bottleneck(input_shape, base_filters=16, bottleneck=64,
                               l2_reg=1e-5, bottleneck_l1=1e-5):
    reg = keras.regularizers.l2(l2_reg) if l2_reg and l2_reg > 0 else None
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(
        bottleneck, 1, use_bias=False,
        kernel_regularizer=keras.regularizers.l1(bottleneck_l1) if bottleneck_l1 else None,
        name="bottleneck_conv"
    )(inputs)
    bn_b = layers.BatchNormalization(name="bottleneck_bn")(x)
    x = layers.Activation("relu")(bn_b)

    c1, p1 = encoder_block(x,  base_filters,     reg)
    c2, p2 = encoder_block(p1, base_filters*2,   reg)
    c3, p3 = encoder_block(p2, base_filters*4,   reg)
    c4, p4 = encoder_block(p3, base_filters*8,   reg)

    bn = conv_block(p4, base_filters*16, reg)
    if CONFIG.get("USE_DUAL_CONTEXT", False):
        bn = aspp(bn, base_filters*16, reg, rates=(1, 6, 12, 18))
    elif CONFIG.get("USE_ASPP", False):
        bn = aspp(bn, base_filters*16, reg, rates=(1, 6, 12, 18))
    elif CONFIG.get("USE_TRANSFORMER_BOTTLENECK", False):
        bn = light_uformer_block(
            bn,
            num_heads=int(CONFIG.get("TRANSFORMER_NUM_HEADS", 4)),
            key_dim=int(CONFIG.get("TRANSFORMER_KEY_DIM", 32)),
            mlp_mult=int(CONFIG.get("TRANSFORMER_MLP_MULT", 2)),
        )
    else:
        bn = layers.Conv2D(base_filters*16, 3, padding="same", dilation_rate=2, kernel_regularizer=reg)(bn)
        bn = layers.BatchNormalization()(bn)
        bn = layers.Activation("relu")(bn)

    d4 = decoder_block(bn, c4, base_filters*8, reg)
    d3 = decoder_block(d4, c3, base_filters*4, reg)
    d2 = decoder_block(d3, c2, base_filters*2, reg)
    d1 = decoder_block(d2, c1, base_filters,   reg)

    H, W = input_shape[0], input_shape[1]
    if CONFIG.get("DEEP_SUPERVISION", False):
        out_main = make_head(d1, 1, name="out_main", out_size=(H, W))
        out_d2   = make_head(d2, 1, name="out_d2",   out_size=(H, W))
        out_d3   = make_head(d3, 1, name="out_d3",   out_size=(H, W))
        out_d4   = make_head(d4, 1, name="out_d4",   out_size=(H, W))
        outputs  = [out_main, out_d2, out_d3, out_d4]
    else:
        outputs = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="out_main")(d1)

    return keras.Model(inputs, outputs, name="UNet_bottleneck")


# ------------- UNet 3+ helpers & builder -------------
def _proj(x, filters, reg, name=None):
    x = layers.Conv2D(filters, 1, padding="same", use_bias=False, kernel_regularizer=reg, name=name)(x)
    x = _Norm()(x)
    x = layers.Activation("relu")(x)
    return x


def _resize_to(x, target, interp="bilinear"):
    H = int(target.shape[1]); W = int(target.shape[2])
    return layers.Resizing(H, W, interpolation=interp)(x)


def _unet3p_fuse(target_grid, sources, filters, reg):
    fused = []
    for s in sources:
        z = s
        if (z.shape[1] != target_grid.shape[1]) or (z.shape[2] != target_grid.shape[2]):
            z = _resize_to(z, target_grid)
        z = _proj(z, filters, reg)
        fused.append(z)
    y = layers.Concatenate()(fused)
    y = conv_block(y, filters, reg, pdrop=0.10)
    return y


def build_unet3p(input_shape, base_filters=16, bottleneck=64,
                 l2_reg=1e-5, bottleneck_l1=1e-5):
    reg = keras.regularizers.l2(l2_reg) if l2_reg and l2_reg > 0 else None
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(
        bottleneck, 1, use_bias=False,
        kernel_regularizer=keras.regularizers.l1(bottleneck_l1) if bottleneck_l1 else None,
        name="bottleneck_conv"
    )(inputs)
    bn_b = layers.BatchNormalization(name="bottleneck_bn")(x)
    x = layers.Activation("relu")(bn_b)

    # Encoder
    c1, p1 = encoder_block(x,  base_filters,     reg)   # H/2
    c2, p2 = encoder_block(p1, base_filters*2,   reg)   # H/4
    c3, p3 = encoder_block(p2, base_filters*4,   reg)   # H/8
    c4, p4 = encoder_block(p3, base_filters*8,   reg)   # H/16

    # Bottleneck
    bn = conv_block(p4, base_filters*16, reg)
    if CONFIG.get("USE_DUAL_CONTEXT", False):
        bn = aspp(bn, base_filters*16, reg, rates=(1, 6, 12, 18))
    elif CONFIG.get("USE_ASPP", False):
        bn = aspp(bn, base_filters*16, reg, rates=(1, 6, 12, 18))
    elif CONFIG.get("USE_TRANSFORMER_BOTTLENECK", False):
        bn = light_uformer_block(
            bn,
            num_heads=int(CONFIG.get("TRANSFORMER_NUM_HEADS", 4)),
            key_dim=int(CONFIG.get("TRANSFORMER_KEY_DIM", 32)),
            mlp_mult=int(CONFIG.get("TRANSFORMER_MLP_MULT", 2)),
        )
    else:
        bn = layers.Conv2D(base_filters*16, 3, padding="same", dilation_rate=2, kernel_regularizer=reg)(bn)
        bn = layers.BatchNormalization()(bn)
        bn = layers.Activation("relu")(bn)

    # UNet 3+ decoder (full-scale fusion)
    t4 = layers.Conv2DTranspose(base_filters*8, 2, strides=2, padding="same")(bn)
    d4 = _unet3p_fuse(t4, [c1, c2, c3, c4, bn], base_filters*8, reg)

    t3 = layers.Conv2DTranspose(base_filters*4, 2, strides=2, padding="same")(d4)
    d3 = _unet3p_fuse(t3, [c1, c2, c3, c4, bn, d4], base_filters*4, reg)

    t2 = layers.Conv2DTranspose(base_filters*2, 2, strides=2, padding="same")(d3)
    d2 = _unet3p_fuse(t2, [c1, c2, c3, c4, bn, d4, d3], base_filters*2, reg)

    t1 = layers.Conv2DTranspose(base_filters,   2, strides=2, padding="same")(d2)
    d1 = _unet3p_fuse(t1, [c1, c2, c3, c4, bn, d4, d3, d2], base_filters, reg)

    d0 = layers.Conv2DTranspose(base_filters, 2, strides=2, padding="same")(d1)

    H, W = input_shape[0], input_shape[1]
    if CONFIG.get("DEEP_SUPERVISION", False):
        out_main = make_head(d0, 1, name="out_main", out_size=(H, W))
        out_d2   = make_head(d2, 1, name="out_d2",   out_size=(H, W))
        out_d3   = make_head(d3, 1, name="out_d3",   out_size=(H, W))
        out_d4   = make_head(d4, 1, name="out_d4",   out_size=(H, W))
        outputs  = [out_main, out_d2, out_d3, out_d4]
    else:
        outputs = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="out_main")(d0)

    return keras.Model(inputs, outputs, name="UNet3Plus")


# ---- Lightweight U-Transformer block (optional bottleneck) ----
def light_uformer_block(x, num_heads=4, key_dim=32, mlp_mult=2):
    """
    Compact transformer on the feature grid.
    Keeps spatial size and channels.
    """
    ch = int(x.shape[-1])

    y = _Norm()(x)
    y = layers.DepthwiseConv2D(3, padding="same", use_bias=False)(y)
    y = layers.Conv2D(ch, 1, use_bias=False)(y)

    H, W = int(y.shape[1]), int(y.shape[2])
    seq = layers.Reshape((H * W, ch))(y)

    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, dropout=0.0)(seq, seq)
    seq = layers.Add()([seq, attn])
    seq = layers.LayerNormalization(epsilon=1e-6)(seq)

    mlp = layers.Dense(ch * mlp_mult, activation="gelu")(seq)
    mlp = layers.Dense(ch)(mlp)
    seq = layers.Add()([seq, mlp])
    seq = layers.LayerNormalization(epsilon=1e-6)(seq)

    y = layers.Reshape((H, W, ch))(seq)
    out = layers.Add()([x, y])
    return out


# ------------- ResNet50 N-channel encoder -------------
def get_resnet50_encoder_nch(x_in, freeze=True):
    """
    ResNet50 with N-channel input by inflating the first conv from RGB.
    Taps: c1=H/2, c2=H/4, c3=H/8, c4=H/16, p4=H/32.
    Assumes ResNet50 is imported from tensorflow.keras.applications.
    """
    base_rgb = ResNet50(include_top=False, weights="imagenet")

    in_ch = int(x_in.shape[-1])
    InputN = layers.Input(shape=(None, None, in_ch))
    base_n = ResNet50(include_top=False, weights=None, input_tensor=InputN)

    rgb_by_name = {l.name: l for l in base_rgb.layers}
    for l in base_n.layers:
        if l.name in rgb_by_name:
            w_src, w_dst = rgb_by_name[l.name].get_weights(), l.get_weights()
            if w_src and w_dst and all(s.shape == d.shape for s, d in zip(w_src, w_dst)):
                l.set_weights(w_src)

    conv_rgb = base_rgb.get_layer("conv1_conv")
    conv_n   = base_n.get_layer("conv1_conv")
    W_rgb, b_rgb = conv_rgb.get_weights()  # (7,7,3,out)
    W_mean = np.mean(W_rgb, axis=2, keepdims=True)
    W_new  = np.repeat(W_mean, in_ch, axis=2) * (3.0 / float(in_ch))
    conv_n.set_weights([W_new, b_rgb])

    c1_t = base_n.get_layer("conv1_relu").output
    c2_t = base_n.get_layer("conv2_block3_out").output
    c3_t = base_n.get_layer("conv3_block4_out").output
    c4_t = base_n.get_layer("conv4_block6_out").output
    p4_t = base_n.output

    feat = keras.Model(base_n.input, [c1_t, c2_t, c3_t, c4_t, p4_t], name="resnet50N_feat")
    if freeze:
        feat.trainable = False
    return feat(x_in)


def build_unet_with_pretrained_encoder(
    input_shape,
    encoder_name="resnet50",
    base_filters=16,
    l2_reg=1e-5,
    freeze=True,
    bottleneck=64,
    bottleneck_l1=1e-5,
):
    """
    Encoder = ResNet50 (inflated to N channels); Decoder = attentioned UNet decoder.
    """
    enc = str(encoder_name).lower()
    if enc not in ("resnet50", "resnet-50"):
        raise ValueError("Only ResNet50 is supported as pretrained encoder in this script.")

    reg = keras.regularizers.l2(l2_reg) if l2_reg and l2_reg > 0 else None
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(
        bottleneck, 1, use_bias=False,
        kernel_regularizer=keras.regularizers.l1(bottleneck_l1) if bottleneck_l1 else None,
        name="bottleneck_conv"
    )(inputs)
    bn_b = layers.BatchNormalization(name="bottleneck_bn")(x)
    x = layers.Activation("relu")(bn_b)

    c1, c2, c3, c4, p4 = get_resnet50_encoder_nch(x, freeze=freeze)

    bn = conv_block(p4, base_filters*16, reg)
    if CONFIG.get("USE_DUAL_CONTEXT", False):
        bn = aspp(bn, base_filters*16, reg, rates=(1, 6, 12, 18))
    elif CONFIG.get("USE_ASPP", False):
        bn = aspp(bn, base_filters*16, reg, rates=(1, 6, 12, 18))
    elif CONFIG.get("USE_TRANSFORMER_BOTTLENECK", False):
        bn = light_uformer_block(
            bn,
            num_heads=int(CONFIG.get("TRANSFORMER_NUM_HEADS", 4)),
            key_dim=int(CONFIG.get("TRANSFORMER_KEY_DIM", 32)),
            mlp_mult=int(CONFIG.get("TRANSFORMER_MLP_MULT", 2)),
        )
    else:
        bn = layers.Conv2D(base_filters*16, 3, padding="same", dilation_rate=2, kernel_regularizer=reg)(bn)
        bn = layers.BatchNormalization()(bn)
        bn = layers.Activation("relu")(bn)

    d4 = decoder_block(bn, c4, base_filters*8, reg)
    d3 = decoder_block(d4, c3, base_filters*4, reg)
    d2 = decoder_block(d3, c2, base_filters*2, reg)
    d1 = decoder_block(d2, c1, base_filters,   reg)

    H, W = input_shape[0], input_shape[1]
    if CONFIG.get("DEEP_SUPERVISION", False):
        out_main = make_head(d1, 1, name="out_main", out_size=(H, W))
        out_d2   = make_head(d2, 1, name="out_d2",   out_size=(H, W))
        out_d3   = make_head(d3, 1, name="out_d3",   out_size=(H, W))
        out_d4   = make_head(d4, 1, name="out_d4",   out_size=(H, W))
        outputs  = [out_main, out_d2, out_d3, out_d4]
    else:
        outputs = layers.Conv2D(1, 1, activation="sigmoid", dtype="float32", name="out_main")(d1)

    return keras.Model(inputs, outputs, name=f"UNet_{encoder_name}")


# ------------- Central builders -------------
def build_model_from_config(input_shape):
    """
    Choose model variant from CONFIG:
    - USE_PRETRAINED_ENCODER: use ResNet50 encoder (inflated to N channels)
    - MODEL_VARIANT: "unet3p" vs baseline UNet
    """
    variant = str(CONFIG.get("MODEL_VARIANT", "unet")).lower()
    use_pre = bool(CONFIG.get("USE_PRETRAINED_ENCODER", False))

    if use_pre:
        return build_unet_with_pretrained_encoder(
            input_shape,
            encoder_name=str(CONFIG.get("ENCODER_BACKBONE", "resnet50")),
            base_filters=CONFIG["BASE_FILTERS"],
            l2_reg=CONFIG["L2_REG"],
            freeze=bool(CONFIG.get("FREEZE_ENCODER", True)),
            bottleneck=CONFIG["BOTTLENECK_CHANNELS"],
            bottleneck_l1=CONFIG["BOTTLENECK_L1"],
        )

    if variant == "unet3p":
        return build_unet3p(
            input_shape,
            base_filters=CONFIG["BASE_FILTERS"],
            bottleneck=CONFIG["BOTTLENECK_CHANNELS"],
            l2_reg=CONFIG["L2_REG"],
            bottleneck_l1=CONFIG["BOTTLENECK_L1"],
        )

    return build_unet_with_bottleneck(
        input_shape,
        base_filters=CONFIG["BASE_FILTERS"],
        bottleneck=CONFIG["BOTTLENECK_CHANNELS"],
        l2_reg=CONFIG["L2_REG"],
        bottleneck_l1=CONFIG["BOTTLENECK_L1"],
    )

## Configure the optimizer and learning-rate schedule

In [ ]:
# =====================================================
# 12) Optimizer / schedule (warmup + cosine)
# =====================================================
init_lr = float(CONFIG.get("LEARNING_RATE", 3e-4))
steps_total = max(1, steps_per_epoch) * int(CONFIG.get("EPOCHS_AFTER_PRUNE", 20))
warmup_steps = max(1, int(0.05 * steps_total))  # 5% warmup


class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, total_steps):
        self.base_lr = float(base_lr)
        self.warmup_steps = int(warmup_steps)
        self.total_steps = int(total_steps)
        self._pi = tf.constant(np.pi, dtype=tf.float32)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = tf.cast(self.warmup_steps, tf.float32)
        total = tf.cast(tf.maximum(1, self.total_steps), tf.float32)
        base  = tf.cast(self.base_lr, tf.float32)

        # linear warmup
        warmup_lr = base * (step / tf.maximum(1.0, warm))

        # cosine decay (after warmup)
        progress = (step - warm) / tf.maximum(1.0, total - warm)
        progress = tf.clip_by_value(progress, 0.0, 1.0)
        cosine_lr = 0.5 * base * (1.0 + tf.cos(self._pi * progress))

        return tf.where(step < warm, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            "base_lr": self.base_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
        }

    @classmethod
    def from_config(cls, config):
        return cls(**config)


lr_sched = WarmupCosine(init_lr, warmup_steps, steps_total)


def build_optimizer(lr_schedule):
    return keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4)

## Define the loss functions

In [ ]:
# =====================================================
# 13) Losses (weighted BCE + per-sample Dice)
# =====================================================
pos_w = 0.5 / max(1e-6, pos_frac)
neg_w = 0.5 / max(1e-6, neg_frac)


@tf.function
def dice_loss(y_true, y_pred, eps=1e-6):
    """
    Per-sample Dice, then averaged across batch.
    Assumes y_true, y_pred have shape (B, H, W, C).
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    inter = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    denom = tf.reduce_sum(y_true + y_pred, axis=[1, 2, 3])

    dice = (2.0 * inter + eps) / (denom + eps)
    return 1.0 - tf.reduce_mean(dice)


@tf.function
def weighted_bce(y_true, y_pred, eps=1e-7):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), eps, 1.0 - eps)
    loss = -(pos_w * y_true * tf.math.log(y_pred) +
             neg_w * (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    return tf.reduce_mean(loss)


@tf.function
def bce_dice_loss(y_true, y_pred, alpha=0.5):
    return alpha * weighted_bce(y_true, y_pred) + (1.0 - alpha) * dice_loss(y_true, y_pred)


@tf.function
def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75, eps=1e-7):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), eps, 1.0 - eps)
    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1.0 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1.0 - y_pred))
    tversky = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return tf.pow(1.0 - tversky, gamma)


_mode = CONFIG.get("LOSS_MODE", "bce_dice").lower()
if _mode == "focal_tversky":
    loss_fn = focal_tversky_loss
elif _mode == "lovasz_hinge":
    raise ValueError("LOSS_MODE='lovasz_hinge' is not implemented in this cleaned script.")
else:
    loss_fn = bce_dice_loss  # default


def make_ds_losses(base_loss, weights):
    """
    Deep supervision loss & metrics dicts.
    """
    out_names = ["out_main", "out_d2", "out_d3", "out_d4"]
    losses = {}
    loss_weights = {}
    metrics = {}
    for i, name in enumerate(out_names):
        losses[name] = base_loss
        loss_weights[name] = float(weights[i]) if i < len(weights) else 0.0
        metrics[name] = [
            keras.metrics.BinaryAccuracy(name=f"{name}_acc"),
            keras.metrics.Precision(name=f"{name}_precision"),
            keras.metrics.Recall(name=f"{name}_recall"),
        ]
    return losses, loss_weights, metrics


def compile_model_from_config(model, opt):
    """
    DS-aware compile when DEEP_SUPERVISION=True; single-output otherwise.
    """
    base_loss = loss_fn
    if CONFIG.get("LOSS_MODE", "bce_dice").lower() == "bce_dice":
        base_loss = bce_dice_loss

    if CONFIG.get("DEEP_SUPERVISION", False):
        losses, loss_wts, metrics = make_ds_losses(
            base_loss, CONFIG.get("DS_LOSS_WEIGHTS", [1.0, 0.5, 0.25, 0.125])
        )
        model.compile(optimizer=opt, loss=losses, loss_weights=loss_wts, metrics=metrics)
    else:
        model.compile(
            optimizer=opt, loss=base_loss,
            metrics=[
                keras.metrics.BinaryAccuracy(name="acc"),
                keras.metrics.Precision(name="precision"),
                keras.metrics.Recall(name="recall"),
            ],
        )


# ---- build model & optimizer ----
NUM_CHANNELS = len(selected_files)
input_shape = (CONFIG["PATCH_SIZE"], CONFIG["PATCH_SIZE"], NUM_CHANNELS)

model = build_model_from_config(input_shape)
opt   = build_optimizer(lr_sched)
compile_model_from_config(model, opt)

print("Model ready.")
try:
    model.summary()
except Exception:
    pass

## Set callbacks and training helpers

In [ ]:
# =====================================================
# 14) Callbacks & training helpers
# =====================================================
LOCAL_CKPT = "/content/best_model.keras"


class BatchGCCallback(keras.callbacks.Callback):
    def __init__(self, every_n_batches=50):
        super().__init__()
        self.n = int(every_n_batches)
        self._b = 0

    def on_train_batch_end(self, batch, logs=None):
        self._b += 1
        if self._b % self.n == 0:
            gc.collect()


base_callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=LOCAL_CKPT,
        save_best_only=True,
        save_weights_only=False,
        monitor="val_loss" if val_steps > 0 else "loss",
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss" if val_steps > 0 else "loss",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.TerminateOnNaN(),
    BatchGCCallback(every_n_batches=50),
]


def _maybe_wrap_targets(batch):
    """
    If DEEP_SUPERVISION is enabled, duplicate y to each named output.
    Expects a batch shaped like (Xb, yb).
    """
    if not CONFIG.get("DEEP_SUPERVISION", False):
        return batch

    Xb, yb = batch
    Y = {
        "out_main": yb,
        "out_d2":   yb,
        "out_d3":   yb,
        "out_d4":   yb,
    }
    return Xb, Y


class DSWrapSequence(keras.utils.Sequence):
    """
    Wrapper that applies _maybe_wrap_targets to each batch.
    """
    def __init__(self, base_seq):
        self.base = base_seq

    def __len__(self):
        return len(self.base)

    def on_epoch_end(self):
        if hasattr(self.base, "on_epoch_end"):
            self.base.on_epoch_end()

    def __getitem__(self, idx):
        return _maybe_wrap_targets(self.base[idx])


def _fit_once(
    model,
    train_input,
    val_input=None,
    epochs=1,
    callbacks=None,
    steps_per_epoch=None,
    val_steps=None,
    tag=None,
):
    """
    Single fit run (no pruning, no retry).
    DS-aware: wraps targets when needed.
    """
    callbacks = callbacks or []
    if tag:
        print(f"Training phase: {tag}")

    # Wrap inputs when deep supervision is ON
    if CONFIG.get("DEEP_SUPERVISION", False):
        if isinstance(train_input, keras.utils.Sequence):
            train_wrapped = DSWrapSequence(train_input)
        elif isinstance(train_input, tf.data.Dataset):
            train_wrapped = train_input.map(
                lambda xb, yb: _maybe_wrap_targets((xb, yb)),
                num_parallel_calls=tf.data.AUTOTUNE,
            )
        else:
            train_wrapped = _maybe_wrap_targets(train_input)

        if val_input is not None:
            if isinstance(val_input, keras.utils.Sequence):
                val_wrapped = DSWrapSequence(val_input)
            elif isinstance(val_input, tf.data.Dataset):
                val_wrapped = val_input.map(
                    lambda xb, yb: _maybe_wrap_targets((xb, yb)),
                    num_parallel_calls=tf.data.AUTOTUNE,
                )
            else:
                val_wrapped = _maybe_wrap_targets(val_input)
        else:
            val_wrapped = None
    else:
        train_wrapped = train_input
        val_wrapped   = val_input

    fit_kwargs = dict(
        x=train_wrapped,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )
    if steps_per_epoch is not None:
        fit_kwargs["steps_per_epoch"] = int(steps_per_epoch)

    if val_wrapped is not None:
        fit_kwargs["validation_data"] = val_wrapped
        if val_steps is not None and val_steps > 0:
            fit_kwargs["validation_steps"] = int(val_steps)

    history = model.fit(**fit_kwargs)
    return history

## Train the model and calibrate the validation threshold

In [ ]:
# =====================================================
# 15) Main training loop (no pruning / warmup stages)
# =====================================================

total_epochs = int(CONFIG["EPOCHS_AFTER_PRUNE"])  # main training epochs

# IMPORTANT: run a single continuous .fit() so EarlyStopping/ModelCheckpoint
# maintain state across all epochs (chunked .fit() calls reset callback state).
_fit_once(
    model,
    train_input=train_seq,
    val_input=val_seq,
    epochs=total_epochs,
    callbacks=base_callbacks,
    steps_per_epoch=steps_per_epoch,
    val_steps=val_steps,
    tag=f"MAIN_1-{total_epochs}",
)

print("Model training has finished. The notebook will now copy the best checkpoint and calculate evaluation metrics.")

# Copy best model once to Drive
try:
    final_ckpt = os.path.join(CONFIG["OUT_DIR"], f"{CONFIG['SAVE_EXPERIMENT_TAG']}_best.keras")
    shutil.copy2(LOCAL_CKPT, final_ckpt)
    print("Best model copied to:", final_ckpt)
except Exception as e:
    print("Could not copy best model to Drive:", e)

## Train the model and calibrate the validation threshold

In [ ]:
# -------------------------
# 15) Validation threshold calibration (use full val set + metrics)
# -------------------------

if val_seq is not None and len(val_seq) > 0:
    print("Collecting probabilities from the full validation set for threshold calibration and evaluation.")
    y_true_all, probs_all = [], []
    for i in tqdm(range(len(val_seq)), desc="Val batches"):
        Xb, yb = val_seq[i]
        pb = model.predict(Xb, verbose=0)

        # Handle deep supervision / dict outputs
        if isinstance(pb, (list, tuple)):
            pb = pb[0]  # main head
        elif isinstance(pb, dict):
            pb = pb.get("out_main", next(iter(pb.values())))

        y_true_all.append((yb.reshape(-1) > 0.5).astype(np.uint8))
        probs_all.append(pb.reshape(-1))

    if y_true_all:
        y_true_all = np.concatenate(y_true_all)
        probs_all  = np.concatenate(probs_all)

        # ---------- Threshold-free metrics ----------
        try:
            roc_auc = roc_auc_score(y_true_all, probs_all)
        except Exception:
            roc_auc = float("nan")
        try:
            ap = average_precision_score(y_true_all, probs_all)  # PR-AUC
        except Exception:
            ap = float("nan")
        try:
            brier = brier_score_loss(y_true_all, probs_all)
        except Exception:
            brier = float("nan")

        print(f" Val (threshold-free): ROC-AUC={roc_auc:.3f}, PR-AUC={ap:.3f}, Brier={brier:.4f}")

        # ---------- Calibration: ECE ----------
        def _expected_calibration_error(y_true, y_proba, n_bins=15):
            y_true = y_true.astype(np.float32)
            y_proba = y_proba.astype(np.float32)

            bins = np.linspace(0.0, 1.0, n_bins + 1)
            binids = np.digitize(y_proba, bins) - 1  # 0..n_bins-1

            ece = 0.0
            N = float(len(y_true))
            if N == 0:
                return 0.0

            for b in range(n_bins):
                mask = binids == b
                if not np.any(mask):
                    continue
                p_true = y_true[mask].mean()
                p_pred = y_proba[mask].mean()
                w = float(mask.sum()) / N
                ece += w * abs(p_pred - p_true)
            return float(ece)

        ece = _expected_calibration_error(y_true_all, probs_all, n_bins=15)
        print(f" Val calibration: ECE={ece:.4f}")

        # ---------- Choose threshold ----------
        mode = CONFIG.get("EVAL_THRESHOLD_MODE", "fixed").lower()
        if mode == "calibrated":
            ths = np.linspace(0.2, 0.8, 25)
            best, best_t = -1.0, 0.5
            for t in ths:
                pred = (probs_all >= t).astype(np.uint8)
                f1 = f1_score(y_true_all, pred, zero_division=0)
                if f1 > best:
                    best, best_t = f1, t
            CONFIG["INFER_THRESHOLD"] = float(best_t)
            print(f" Using calibrated threshold from validation: best_t={best_t:.3f}, best F1={best:.3f}")
        else:
            # Fixed threshold for fair model-to-model comparison
            CONFIG["INFER_THRESHOLD"] = float(CONFIG.get("FIXED_THRESHOLD", 0.5))
            print(f" Using FIXED threshold for evaluation: t={CONFIG['INFER_THRESHOLD']:.3f}")

        # Always print the final threshold clearly
        t = float(CONFIG["INFER_THRESHOLD"])
        print(f" Final validation threshold in use: t={t:.3f}")

        # ---------- Thresholded metrics on validation ----------
        y_pred_all = (probs_all >= t).astype(np.uint8)

        acc  = accuracy_score(y_true_all, y_pred_all)
        prec = precision_score(y_true_all, y_pred_all, zero_division=0)
        rec  = recall_score(y_true_all, y_pred_all, zero_division=0)
        f1   = f1_score(y_true_all, y_pred_all, zero_division=0)
        iou  = jaccard_score(y_true_all, y_pred_all, zero_division=0)

        try:
            mcc = matthews_corrcoef(y_true_all, y_pred_all)
        except Exception:
            mcc = float("nan")

        # Confusion matrix (handle edge cases where a class might be missing)
        try:
            cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
            if cm.shape == (2, 2):
                tn, fp, fn, tp = cm.ravel()
            else:
                tn = fp = fn = tp = None
        except Exception:
            cm = None
            tn = fp = fn = tp = None

        print(
            f" Val @ t={t:.3f}: "
            f"acc={acc:.3f}, prec={prec:.3f}, rec={rec:.3f}, "
            f"f1={f1:.3f}, IoU={iou:.3f}, MCC={mcc:.3f}"
        )
        if cm is not None and tn is not None:
            print(
                f"   Confusion matrix (val, t={t:.3f}): "
                f"TN={tn}, FP={fp}, FN={fn}, TP={tp}"
            )

    else:
        print(" No validation batches produced.")
else:
    print("Validation threshold calibration was skipped because no validation patches were available.")

## Evaluate the independent test set

In [ ]:
# -------------------------
# 15B) Final TEST evaluation (independent of early stopping + threshold tuning)
# -------------------------
if "test_seq" in globals() and test_seq is not None and len(test_seq) > 0:
    print("Collecting probabilities from the independent test set for final evaluation.")
    y_true_all, probs_all = [], []
    for i in tqdm(range(len(test_seq)), desc="Test batches"):
        Xb, yb = test_seq[i]
        pb = model.predict(Xb, verbose=0)

        # Handle deep supervision / dict outputs
        if isinstance(pb, (list, tuple)):
            pb = pb[0]  # main head
        elif isinstance(pb, dict):
            pb = pb.get("out_main", next(iter(pb.values())))

        y_true_all.append((yb.reshape(-1) > 0.5).astype(np.uint8))
        probs_all.append(pb.reshape(-1))

    if y_true_all:
        y_true_all = np.concatenate(y_true_all)
        probs_all  = np.concatenate(probs_all)

        # Threshold-free metrics
        try:
            roc_auc = roc_auc_score(y_true_all, probs_all)
        except Exception:
            roc_auc = float("nan")
        try:
            ap = average_precision_score(y_true_all, probs_all)  # PR-AUC
        except Exception:
            ap = float("nan")
        try:
            brier = brier_score_loss(y_true_all, probs_all)
        except Exception:
            brier = float("nan")
        print(f" TEST (threshold-free): ROC-AUC={roc_auc:.3f}, PR-AUC={ap:.3f}, Brier={brier:.4f}")

        # Thresholded metrics using the already-chosen threshold (do NOT re-tune on test)
        t = float(CONFIG.get("INFER_THRESHOLD", 0.5))
        y_pred_all = (probs_all >= t).astype(np.uint8)

        acc  = accuracy_score(y_true_all, y_pred_all)
        prec = precision_score(y_true_all, y_pred_all, zero_division=0)
        rec  = recall_score(y_true_all, y_pred_all, zero_division=0)
        f1   = f1_score(y_true_all, y_pred_all, zero_division=0)
        iou  = jaccard_score(y_true_all, y_pred_all, zero_division=0)

        try:
            mcc = matthews_corrcoef(y_true_all, y_pred_all)
        except Exception:
            mcc = float("nan")

        try:
            cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])
            if cm.shape == (2, 2):
                tn, fp, fn, tp = cm.ravel()
            else:
                tn = fp = fn = tp = None
        except Exception:
            cm = None
            tn = fp = fn = tp = None

        print(
            f" TEST @ t={t:.3f}: "
            f"acc={acc:.3f}, prec={prec:.3f}, rec={rec:.3f}, "
            f"f1={f1:.3f}, IoU={iou:.3f}, MCC={mcc:.3f}"
        )
        if cm is not None and tn is not None:
            print(
                f"   Confusion matrix (test, t={t:.3f}): "
                f"TN={tn}, FP={fp}, FN={fn}, TP={tp}"
            )
else:
    print("Test evaluation was skipped because no test patches were available.")

## Generate seamless full-scene probability rasters

In [ ]:
# -------------------------
# 16) Full-scene prediction (overlap + Hann blending)
# -------------------------

def open_predictors_for_area(area_code):
    """
    Return predictor paths for a given area that match selected_basenames.
    Uses PREDICTOR_SUBDIR_TEMPLATE.
    """
    pred_subdir = CONFIG.get("PREDICTOR_SUBDIR_TEMPLATE", "{area}_layers").format(area=area_code)
    pred_dir = DATA_DIR / pred_subdir
    try:
        predictors = find_predictors_in_subdir(pred_dir)
    except FileNotFoundError as e:
        print(f"    {e}")
        return None

    by_base = {os.path.basename(p): p for p in predictors}
    missing = [b for b in selected_basenames if b not in by_base]
    if missing:
        print(f"    Predictors missing for area {area_code}, skipping prediction.")
        for m in missing:
            print("      -", m)
        return None

    paths = [by_base[b] for b in selected_basenames]
    if CONFIG.get("USE_LOCAL_CACHE", True):
        paths = ensure_local_cache(paths,
                                   CONFIG["LOCAL_CACHE_DIR"],
                                   area_token=area_code)
    return paths

    print("Starting full-scene prediction with overlapping tiles and Hann blending.")

PRED_PATCH  = CONFIG["PATCH_SIZE"]
PRED_OVER   = CONFIG["PRED_OVERLAP_FRACTION"]
PRED_STRIDE = max(1, int(PRED_PATCH * (1.0 - PRED_OVER)))
PRED_STRIDE = min(PRED_PATCH, PRED_STRIDE)

w1d = np.hanning(PRED_PATCH).astype(np.float32)
if w1d.max() == 0:
    w1d = np.ones(PRED_PATCH, dtype=np.float32)
w2d = (np.outer(w1d, w1d).astype(np.float32) + 1e-6)


def iter_windows_for_pred(H, W, patch, stride):
    for rr in inclusive_starts(H, patch, stride):
        for cc in inclusive_starts(W, patch, stride):
            yield Window(cc, rr, patch, patch), (rr, cc)


# ---------- SINGLE-AREA PREDICTION (original behaviour) ----------
if not CONFIG.get("USE_MULTI_AREA", False):
    print(f"Starting full-scene prediction for single area: {CONFIG['AREA_TOKEN']}")

    sum_pred = np.zeros((height, width), dtype=np.float32)
    sum_w    = np.zeros((height,  width), dtype=np.float32)

    for wdw, (r, c) in tqdm(
        iter_windows_for_pred(height, width, PRED_PATCH, PRED_STRIDE),
        desc=f"Predicting {CONFIG['AREA_TOKEN']}"
    ):
        Xw = stack_window_cached(wdw).astype(np.float32)
        Xw = (Xw - CHANNEL_MEAN) / CHANNEL_STD

        pred_full = model.predict(Xw[None, ...], verbose=0)
        if isinstance(pred_full, (list, tuple)):
            pred_full = pred_full[0]
        elif isinstance(pred_full, dict):
            pred_full = pred_full.get("out_main", next(iter(pred_full.values())))

        pw = pred_full[0, ..., 0]
        rr = slice(r, r + PRED_PATCH)
        cc = slice(c, c + PRED_PATCH)
        sum_pred[rr, cc] += pw * w2d
        sum_w[rr, cc]    += w2d

    prob = np.divide(
        sum_pred,
        np.maximum(sum_w, 1e-6),
        out=np.zeros_like(sum_pred),
        where=sum_w > 0,
    )

    pred_out_path = os.path.join(CONFIG["OUT_DIR"], CONFIG["SAVE_PRED_TIF"])
    prof = dict(ref_profile)
    prof.update(count=1, dtype="float32", compress="lzw")
    with rasterio.open(pred_out_path, "w", **prof) as dst:
        dst.write(prob.astype(np.float32), 1)
    print("Full-scene probability raster saved to:", pred_out_path)


# ---------- MULTI-AREA PREDICTION ----------
else:
    print("Starting full-scene prediction for every area listed in AREA_CODES.")

    for area in CONFIG["AREA_CODES"]:
        print(f"\n=============================")
        print(f" Predicting area {area}")
        print(f"=============================")

        # --- CHM as the reference grid/profile for this area ---
        chm_path_a = find_chm_for_area(area)
        if chm_path_a is None:
            print(f"    No CHM found for {area}, skipping prediction.")
            continue

        with rasterio.open(chm_path_a) as ref_a:
            prof_a = ref_a.profile
            H_a, W_a = ref_a.height, ref_a.width

        print(f"   Grid: {W_a}x{H_a}")

        # --- Predictors for this area, matching training feature set ---
        pred_paths_a = open_predictors_for_area(area)
        if pred_paths_a is None:
            continue

        # Open predictor rasters
        ds_paths_a = list(pred_paths_a)
        ds_list_a  = [rasterio.open(p) for p in ds_paths_a]

        def _safe_read_with_retry_a(idx, window):
            tries = int(CONFIG["READ_RETRIES"]) + 1
            for _ in range(tries):
                try:
                    return ds_list_a[idx].read(1, window=window, boundless=False)
                except Exception:
                    try:
                        ds_list_a[idx].close()
                    except:
                        pass
                    ds_list_a[idx] = rasterio.open(ds_paths_a[idx])
            return np.zeros((window.height, window.width), dtype=np.float32)

        def stack_window_cached_a(window):
            bands = []
            for i in range(len(ds_list_a)):
                arr = _safe_read_with_retry_a(i, window)
                arr = np.where(np.isfinite(arr), arr, 0).astype(np.float32)
                bands.append(arr)
            return np.stack(bands, axis=-1)

        # --- Hann-blended tiling prediction for this area ---
        sum_pred = np.zeros((H_a, W_a), dtype=np.float32)
        sum_w    = np.zeros((H_a, W_a), dtype=np.float32)

        for wdw, (r, c) in tqdm(
            iter_windows_for_pred(H_a, W_a, PRED_PATCH, PRED_STRIDE),
            desc=f"Predicting {area}"
        ):
            Xw = stack_window_cached_a(wdw).astype(np.float32)
            Xw = (Xw - CHANNEL_MEAN) / CHANNEL_STD

            pred_full = model.predict(Xw[None, ...], verbose=0)
            if isinstance(pred_full, (list, tuple)):
                pred_full = pred_full[0]
            elif isinstance(pred_full, dict):
                pred_full = pred_full.get("out_main", next(iter(pred_full.values())))

            pw = pred_full[0, ..., 0]
            rr = slice(r, r + PRED_PATCH)
            cc = slice(c, c + PRED_PATCH)
            sum_pred[rr, cc] += pw * w2d
            sum_w[rr, cc]    += w2d

        prob_a = np.divide(
            sum_pred,
            np.maximum(sum_w, 1e-6),
            out=np.zeros_like(sum_pred),
            where=sum_w > 0,
        )

        # Save per-area probability raster
        pred_name = f"{area}_{CONFIG['SAVE_PRED_TIF']}"
        pred_out_path_a = os.path.join(CONFIG["OUT_DIR"], pred_name)
        prof_write = dict(prof_a)
        prof_write.update(count=1, dtype="float32", compress="lzw")
        with rasterio.open(pred_out_path_a, "w", **prof_write) as dst:
            dst.write(prob_a.astype(np.float32), 1)
        print(f"Probability raster for {area} saved to: {pred_out_path_a}")

        # close predictors
        for ds_a in ds_list_a:
            try:
                ds_a.close()
            except:
                pass

print("All requested training, evaluation, and prediction steps are complete.")